# ❄️🐉 cryoDRGN on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab.ipynb)

**cryoDRGN** is a neural-network method for **heterogeneous cryo-EM reconstruction** — it learns a *continuous* distribution of 3D structures directly from a single-particle dataset.

This notebook walks you end-to-end on a free/Pro Colab GPU:

| Step | What happens |
|------|--------------|
| 1. Setup | Check the GPU, install cryoDRGN, mount Google&nbsp;Drive |
| 2. Inputs | Point to your particles, poses and CTF (sourced from Drive) |
| 3. Preprocess | `downsample` → `parse_pose_*` → `parse_ctf_*` |
| 4. Sanity check | `backproject_voxel` a subset and view slices |
| 5. Train | `train_vae` a heterogeneous model |
| 6. Analyze | `analyze` the latent space + view plots and volumes inline |
| 7. Diagnose | Loss/KLD curves and `analyze_convergence` — has training converged? |
| 8. Filter *(optional)* | Remove junk particles and retrain — in Colab, or export for local `cryodrgn filter` |
| 9. Save | Sync results back to your Google Drive |

> **You will need**, from an upstream consensus refinement (RELION or cryoSPARC):
> - a **particle stack** — `.mrcs` / `.star` / `.cs` / `.txt`
> - a **`.star`** (RELION) **or `.cs`** (cryoSPARC) file to extract **poses** and **CTF** from
>
> Don't have data yet? Try the tutorial dataset (EMPIAR-10076) from the
> [cryoDRGN user guide](https://ez-lab.gitbook.io/cryodrgn/).

📖 Docs: <https://ez-lab.gitbook.io/cryodrgn/> · 💻 GitHub: <https://github.com/ml-struct-bio/cryodrgn> · 📄 [Zhong et al., *Nature Methods* 2021](https://doi.org/10.1038/s41592-020-01049-4)

---
### ⚙️ Before you start — turn on the GPU
**Runtime → Change runtime type → Hardware accelerator → GPU** (a T4 is fine for `D=128`; use an A100/L4 on Colab Pro for `D=256`).

Then run the cells **in order** (▶ on each, or *Runtime → Run all*). Each cell is a collapsible **form** — edit the fields on the right, no coding required.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

## 2 · Connect Google Drive

We use **two locations**, which is standard practice for cryo-EM on Colab:

- 📁 **Drive project folder** — *durable* storage for your inputs and final results. Survives disconnects.
- ⚡ **Local scratch** (`/content/...`) — *fast* disk for the downsampled stack that training reads. Wiped when the runtime ends, but mirrored to Drive so it can be restored.

We read inputs from Drive and downsample onto fast local scratch **while mirroring a durable copy (plus `pose.pkl`/`ctf.pkl`) back to Drive**; training then writes its model **directly to Drive** (so per-epoch checkpoints survive a disconnect). The net effect: nothing expensive has to be recomputed after an interruption.

In [ ]:
#@title 2.1 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

In [ ]:
#@title 2.2 · Choose your project folder { display-mode: "form" }
#@markdown **`drive_project_dir`** — a folder in *your* Drive for this project (created if missing).
#@markdown Put your input files here, and final results are saved back here.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown **`local_work_dir`** — fast local scratch where training runs.
local_work_dir = "/content/cryodrgn_work"  #@param {type:"string"}

import os

DRIVE_DIR = os.path.abspath(drive_project_dir)
WORK_DIR = os.path.abspath(local_work_dir)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Persist these across cells for the rest of the notebook.
os.environ["CRYODRGN_DRIVE_DIR"] = DRIVE_DIR
os.environ["CRYODRGN_WORK_DIR"] = WORK_DIR
os.chdir(WORK_DIR)

print(f"📁 Drive project (durable) : {DRIVE_DIR}")
print(f"⚡ Local scratch (fast)    : {WORK_DIR}")
print(f"📂 Working directory       : {os.getcwd()}")
print("\nContents of your Drive project folder:")
for f in sorted(os.listdir(DRIVE_DIR)) or ["(empty — upload your inputs here)"]:
    print("   ", f)

## 3 · Point to your input files

cryoDRGN needs three things, all derived from an upstream **consensus refinement**:

1. **Particle images** — the stack you refined (`.mrcs`, `.star`, `.cs`, or a `.txt` of `.mrcs` paths).
2. **Poses** — orientation + shift per particle, extracted from the refinement's `.star`/`.cs`.
3. **CTF parameters** — extracted from the same `.star`/`.cs`.

Fill in the paths below (they usually live inside your Drive project folder from Step 2).

In [ ]:
#@title 3.1 · Locate inputs on Drive { display-mode: "form" }
import os
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]

#@markdown **Particle stack** — path to your images (`.mrcs`/`.star`/`.cs`/`.txt`). For a cryoSPARC
#@markdown `.cs` or RELION `.star`, point this at that **same file** (reuse it as *metadata_file*).
particles = "/content/drive/MyDrive/cryodrgn_project/particles.cs"  #@param {type:"string"}

#@markdown **Metadata file** — the RELION `.star` **or** cryoSPARC `.cs` that holds poses & CTF
#@markdown (for a `.cs`/`.star` dataset, the same file as *particles* above).
metadata_file = "/content/drive/MyDrive/cryodrgn_project/particles.cs"  #@param {type:"string"}

#@markdown **`datadir`** — the folder the metadata file's internal image paths resolve against.
#@markdown Often needed for cryoSPARC: set it to the export/job folder the `.cs` blob paths are
#@markdown relative to (e.g. `.../J486_particles_0`, so `J486/reconstructed/<uid>_particles.mrc`
#@markdown resolves). Leave blank if the images already resolve next to the metadata file.
datadir = ""  #@param {type:"string"}

# make these available to later cells
os.environ["CRYODRGN_PARTICLES"] = particles
os.environ["CRYODRGN_META"] = metadata_file
os.environ["CRYODRGN_DATADIR"] = datadir

print("Checking inputs...\n")
for label, path in [("Particles", particles), ("Metadata (.star/.cs)", metadata_file)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'} {label}: {path}")
    if not ok:
        print("       ^ not found — fix the path above (check spelling / that it's in your Drive).")
if datadir:
    print(f"  {'✅' if os.path.isdir(datadir) else '❌'} datadir: {datadir}")

## 4 · Preprocess

Three quick commands turn your raw inputs into what `train_vae` expects:
`downsample` the images, then extract `pose.pkl` and `ctf.pkl`. All three are **saved to your
Drive project folder** and **skipped on re-run if already present**, so an interrupted session
resumes without re-generating them (the downsampled stack restores from Drive rather than being
recomputed).

In [ ]:
#@title 4.1 · Downsample the images — runs in the background { display-mode: "form" }
#@markdown Streams your particles from Drive into a smaller stack on **fast local disk**, then
#@markdown mirrors a durable copy to Drive. This reads the **full-size** images from Drive (often
#@markdown 100s of GB), so it is a **multi-hour job** and runs in the **background**: stop the
#@markdown monitor any time with ⏹ and the job keeps going — re-run this cell to re-attach.
box_size = 128  #@param [64, 128, 192, 200, 256] {type:"raw"}
#@markdown Images processed at once. Peak RAM ≈ `batch_size × rawbox² × 32 B` (measured: the
#@markdown Hartley transform holds a complex64 copy of the **raw** box alongside the input), so
#@markdown cryoDRGN's default of 5000 needs ~26 GB at 400×400. It changes nothing about the
#@markdown output and does not make this faster — above ~250 the per-image cost is flat, and on
#@markdown a 2-core runtime large batches are slower. This step is CPU-only; the GPU is idle.
batch_size = 1000  #@param {type:"integer"}
#@markdown Output split into files of this many images (`0` = one big file). Chunking makes the
#@markdown Drive mirror resumable — recommended for large stacks.
chunk = 10000  #@param {type:"integer"}
#@markdown Keep a durable copy of the downsampled stack on Drive (recommended).
backup_to_drive = True  #@param {type:"boolean"}

import os, glob, shutil, time, subprocess
from IPython.display import clear_output

particles = os.environ["CRYODRGN_PARTICLES"]
datadir = os.environ.get("CRYODRGN_DATADIR", "")
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

# cryodrgn downsample rejects an odd box, or one larger than the source
# (downsample.py:242-249) -- but only after opening the stack, so check first. The dropdown
# is {type:"raw"}, so anything can be typed into it.
D_out = int(box_size)
if D_out % 2 != 0:
    raise ValueError(f"box_size must be even (got {D_out}); cryoDRGN's lattice requires it "
                     f"(lattice.py:205).")
try:
    from cryodrgn.source import ImageSource
    _srcD = ImageSource.from_file(particles, lazy=True, datadir=datadir or "").D
except Exception:
    _srcD = None
if _srcD is not None:
    if D_out > _srcD:
        raise ValueError(f"box_size {D_out} exceeds the source box {_srcD} — downsample can "
                         f"only shrink.")
    if D_out == _srcD:
        print(f"ℹ️  box_size {D_out} equals the source box — this is a straight copy, not a "
              f"downsample.")
    print(f"box       : {_srcD} -> {D_out}"
          + ("" if D_out % 8 == 0 else
             f"   (note: {D_out} is not a multiple of 8, so AMP training will warn)"))

    # downsample streams one batch at a time (lazy source, source.py:337-345), so peak RAM is
    # one batch through the Hartley transform: measured at 28 B per RAW pixel, because
    # ht2_center holds a complex64 copy of the raw box alongside the float32 input.
    # 28 B/px measured for the transform chain itself, plus ~4 B/px for the reader's own
    # staging array (MRCFileSource._images allocates np.zeros((B,D,D)) which images() then
    # copies into a tensor, source.py:392).
    _need = int(batch_size) * _srcD * _srcD * 32
    def _free_ram():                       # _avail_ram() is defined further down this cell
        try:
            import psutil
            return psutil.virtual_memory().available
        except Exception:
            try:
                for _ln in open("/proc/meminfo"):
                    if _ln.startswith("MemAvailable"):
                        return int(_ln.split()[1]) * 1024
            except Exception:
                pass
        return 0
    _free = _free_ram()
    print(f"batch RAM : {_need/2**30:.1f} GiB for -b {int(batch_size)} at raw box {_srcD}"
          + (f"   ({_free/2**30:.0f} GiB free)" if _free else ""))
    if _free and _need > _free * 0.7:
        _fits = int(_free * 0.5 / (_srcD * _srcD * 32))
        raise MemoryError(
            f"-b {int(batch_size)} needs {_need/2**30:.1f} GiB but only {_free/2**30:.1f} GiB "
            f"is free.\n  Colab kills the runtime on OOM with no traceback, so this stops "
            f"first. Use -b {max(100, _fits // 100 * 100)} or less.\n  Batch size does not "
            f"change the output, and above ~250 it does not change the speed either.")

stem = f"particles.{box_size}"
local_mrcs = os.path.join(WORK_DIR, stem + ".mrcs")
chunked = bool(chunk and int(chunk) > 0)
# with --chunk, downstream steps read the particles.<D>.txt index, not the .mrcs
local_stack = (os.path.splitext(local_mrcs)[0] + ".txt") if chunked else local_mrcs
drive_stack = os.path.join(DRIVE_DIR, os.path.basename(local_stack))
# The preprocessing notebook may have written the stack into a SUBFOLDER of the project
# dir; look there too rather than silently re-downsampling hundreds of GB.
_sub = sorted(glob.glob(os.path.join(DRIVE_DIR, "*", os.path.basename(local_stack))))
if not os.path.exists(drive_stack) and _sub:
    if len(_sub) == 1:
        drive_stack = _sub[0]
        print("ℹ️  Using the existing stack found in %s" % os.path.dirname(drive_stack))
    else:
        print("⚠️  Stacks found in several subfolders — point drive_project_dir at one of:")
        for _c in _sub:
            print("     %s" % os.path.dirname(_c))
job_sh, job_log = os.path.join(WORK_DIR, stem + ".job.sh"), os.path.join(WORK_DIR, stem + ".log")
job_pid, job_rc = os.path.join(WORK_DIR, stem + ".pid"), os.path.join(WORK_DIR, stem + ".rc")
job_exp = os.path.join(WORK_DIR, stem + ".expect")
DATA_EXT = (".mrcs", ".txt")

def _mirror(src_dir, dst_dir):
    """Copy stack files (plus any --chunk tiles) between dirs, skipping identical ones."""
    os.makedirs(dst_dir, exist_ok=True)
    copied = 0
    for f in sorted(glob.glob(os.path.join(src_dir, stem + "*"))):
        if not f.endswith(DATA_EXT):
            continue
        dst = os.path.join(dst_dir, os.path.basename(f))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
            shutil.copy2(f, dst)
            copied += 1
    return copied

def _live_pid():
    """PID of a still-running downsample job for this box size, else None."""
    try:
        pid = int(open(job_pid).read().strip())
        cmdline = open("/proc/%d/cmdline" % pid, "rb").read().decode("utf8", "ignore")
    except Exception:
        return None
    return pid if "cryodrgn" in cmdline or stem in cmdline else None

def _bytes_out():
    return sum(os.path.getsize(f) for f in glob.glob(os.path.join(WORK_DIR, stem + "*"))
               if f.endswith(DATA_EXT))

def _avail_ram():
    try:
        import psutil
        return psutil.virtual_memory().available
    except Exception:
        try:
            for ln in open("/proc/meminfo"):
                if ln.startswith("MemAvailable"):
                    return int(ln.split()[1]) * 1024
        except Exception:
            pass
    return None

interrupted = False
pid = _live_pid()

if pid:
    print("↻ A downsample job is already running (PID %d) — re-attaching to it." % pid)
elif os.path.exists(local_stack):
    print("✔ Local stack already present — skipping downsample: %s" % local_stack)
elif os.path.exists(drive_stack):
    print("↧ Found the stack on Drive — restoring to local disk (no re-downsample needed)...")
    t0 = time.time(); _mirror(os.path.dirname(drive_stack), WORK_DIR)
    print("  restored in %.0fs" % (time.time() - t0))
else:
    # ---- preflight: will this actually fit in disk and RAM? ----
    n_imgs = raw_D = None
    itemsize = raw_itemsize = 4
    try:
        import numpy as _np
        from cryodrgn.source import ImageSource
        _src = ImageSource.from_file(particles, lazy=True, datadir=datadir or "")
        # downsample preserves the source dtype, so never assume float32 here:
        # cryoSPARC often writes 2-byte images, which halves every size below.
        n_imgs, raw_D = int(_src.n), int(_src.D)
        # Output dtype = what ImageSource reports. A direct .mrc/.mrcs preserves its dtype,
        # but .cs/.star/.txt never pass one through, so they report (and write) float32.
        itemsize = _np.dtype(_src.dtype).itemsize
        raw_itemsize = itemsize
        try:   # on-disk dtype of the underlying files, for the read-volume estimate
            _s = next(v for v in getattr(_src, "_sources", {}).values() if v is not None)
            raw_itemsize = _np.dtype(_s.dtype).itemsize
        except Exception:
            pass
        del _src
    except Exception as e:
        print("⚠️  Could not inspect the input to pre-check sizes (%s: %s)" % (type(e).__name__, e))

    free_local = shutil.disk_usage(WORK_DIR).free
    ram = _avail_ram()
    print("Runtime: %.0f GB free on local disk%s" % (
        free_local / 1e9, ", %.0f GB RAM available" % (ram / 1e9) if ram else ""))

    out_bytes = None
    if n_imgs:
        out_bytes = n_imgs * int(box_size) ** 2 * itemsize
        read_bytes = n_imgs * raw_D ** 2 * raw_itemsize
        print("Input: %s images at %dx%d (%d B/px on disk)  →  output %.1f GB (%d B/px)" % (
            format(n_imgs, ","), raw_D, raw_D, raw_itemsize, out_bytes / 1e9, itemsize))
        print("       reads ~%.0f GB of full-size images from Drive — "
              "expect hours, not minutes." % (read_bytes / 1e9))
        if out_bytes > free_local * 0.97:
            raise MemoryError(
                "Output stack (%.1f GB) does not fit in local free space (%.1f GB). "
                "Use a smaller box_size, or free space first." % (out_bytes / 1e9, free_local / 1e9))
        if out_bytes > free_local * 0.75:
            print("⚠️  This will use most of the local disk — keep an eye on free space.")
        if ram and out_bytes > ram * 0.85:
            print("ℹ️  Note for Step 6: this stack (%.1f GB) exceeds this runtime's RAM, so "
                  "cell 6.1 will\n    switch on --lazy by itself (slower epochs). A High-RAM "
                  "runtime avoids that." % (out_bytes / 1e9))

    if raw_D and ram:
        peak = int(batch_size) * raw_D ** 2 * 20  # f32 chunk + complex64 FFT + shifted copy
        print("Downsample peak RAM at -b %d: ~%.1f GB" % (int(batch_size), peak / 1e9))
        if peak > ram * 0.9:
            raise MemoryError(
                "batch_size %d needs ~%.1f GB but only %.1f GB is available. Lower it (try %d)." % (
                    int(batch_size), peak / 1e9, ram / 1e9,
                    max(100, int(int(batch_size) * ram * 0.5 / peak) // 100 * 100)))
        if peak > ram * 0.6:
            print("⚠️  Tight — consider lowering batch_size if the kernel dies.")

    cmd = 'cryodrgn downsample "%s" -D %d -o "%s" -b %d' % (
        particles, int(box_size), local_mrcs, int(batch_size))
    if chunked:
        cmd += " --chunk %d" % int(chunk)
    if datadir:
        cmd += ' --datadir "%s"' % datadir

    for f in (job_pid, job_rc, job_exp):
        if os.path.exists(f):
            os.remove(f)
    if out_bytes:
        open(job_exp, "w").write(str(out_bytes))
    # Detached (start_new_session) so stopping the monitor — or losing the browser tab —
    # does not kill the job. It writes its own PID, then its exit code when finished.
    with open(job_sh, "w") as f:
        f.write("#!/bin/bash\necho $$ > %s\n%s\necho $? > %s\n" % (job_pid, cmd, job_rc))
    print("\n$", cmd, "\n")
    subprocess.Popen(["bash", job_sh], stdout=open(job_log, "w"),
                     stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                     start_new_session=True, cwd=WORK_DIR)
    for _ in range(60):
        pid = _live_pid()
        if pid:
            break
        time.sleep(0.25)
    print("🚀 Launched in the background (PID %s). Log: %s" % (pid, job_log))

if pid:
    expect = None
    try:
        expect = int(open(job_exp).read().strip())
    except Exception:
        pass
    t0, b0 = time.time(), _bytes_out()
    try:
        while True:
            alive = _live_pid()
            nb, el = _bytes_out(), time.time() - t0
            try:
                tail = [ln for ln in open(job_log).read().strip().split("\n") if ln][-6:]
            except Exception:
                tail = []
            clear_output(wait=True)
            print("⏳ cryodrgn downsample — %s, elapsed %.0f min" % (
                ("running (PID %d)" % alive) if alive else "process finished", el / 60))
            line = "   written: %.2f GB" % (nb / 1e9)
            if expect:
                line += " / ~%.1f GB  (%.0f%%)" % (expect / 1e9, 100.0 * nb / expect)
            print(line)
            rate = (nb - b0) / max(1.0, el)
            if expect and rate > 1e5 and nb < expect:
                print("   rate %.0f MB/s → ETA ~%.0f min" % (
                    rate / 1e6, (expect - nb) / rate / 60))
            if tail:
                print("   --- log tail ---")
                for ln in tail:
                    print("   " + ln[:160])
            print("\n⏹ You can stop this cell any time — the job keeps running.")
            print("   Re-run this cell to re-attach; other cells are free while it runs.")
            if not alive:
                break
            time.sleep(15)
    except KeyboardInterrupt:
        interrupted = True
        print("\n⏹ Monitor detached — the downsample job is STILL RUNNING in the background.")
        print("   Re-run this cell to re-attach and watch progress.")

if not interrupted:
    if os.path.exists(job_rc):
        rc = int(open(job_rc).read().strip() or 1)
        if rc != 0:
            raise RuntimeError("downsample failed (exit %d) — see the log: %s" % (rc, job_log))
    if not os.path.exists(local_stack):
        raise RuntimeError("Expected %s but it was not created — check %s" % (local_stack, job_log))

    if backup_to_drive:
        sz = _bytes_out()
        already = sum(os.path.getsize(f) for f in glob.glob(os.path.join(DRIVE_DIR, stem + "*"))
                      if f.endswith(DATA_EXT))
        try:
            drive_free = shutil.disk_usage(DRIVE_DIR).free
        except Exception:
            drive_free = None
        if drive_free is not None and (sz - already) > drive_free:
            print("⚠️  Skipping Drive mirror: need %.1f GB but only %.1f GB free on Drive." % (
                (sz - already) / 1e9, drive_free / 1e9))
        else:
            print("⇪ Mirroring stack to Drive (%.1f GB, one-time — lets a future session resume "
                  "without re-downsampling)..." % (sz / 1e9))
            t0 = time.time(); n = _mirror(WORK_DIR, DRIVE_DIR)
            print("  %s in %.0fs" % (("copied %d file(s)" % n) if n else "already up to date",
                                     time.time() - t0))

    os.environ["CRYODRGN_DOWNSAMPLED"] = local_stack
    print("\n✅ Training stack (local, fast): %s" % local_stack)
    if backup_to_drive:
        print("✅ Durable copy on Drive:        %s" % drive_stack)

In [ ]:
#@title 4.2 · Parse poses → pose.pkl (saved to Drive) { display-mode: "form" }
#@markdown Written straight to your Drive project folder (small file) and skipped on re-run if it
#@markdown already exists — so you never re-parse after a disconnect. **Re-running this is always
#@markdown safe:** with `pose.pkl` present it parses nothing and overwrites nothing, it only
#@markdown exports the path that 6.1 / 6.2 / 8.3 read. Those cells REQUIRE it, so run this in any
#@markdown session where you intend to train, even one where the stack is already prepared.
pose_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown **`box_size_D`** — box size of the **consensus refinement** (the *original*, un-downsampled
#@markdown images). A recent cryoSPARC `.cs` carries this so `0` (auto) usually works; `.star` is
#@markdown auto-detected if present. Set it only if parsing fails.
box_size_D = 0  #@param {type:"integer"}

import os
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
pose_pkl = os.path.join(DRIVE_DIR, "pose.pkl")
os.environ["CRYODRGN_POSE"] = pose_pkl

if os.path.exists(pose_pkl):
    # Nothing is re-parsed and nothing is overwritten; this run only exports the path so
    # that 6.1 / 6.2 / 8.3 can find it. Safe to run in any session, in any order.
    print(f"✔ Poses already on Drive — skipping parse: {pose_pkl}")
    print(f"   ({os.path.getsize(pose_pkl)/2**20:.1f} MB, last modified "
          f"{__import__('time').ctime(os.path.getmtime(pose_pkl))})")
else:
    # Only needed when we actually have to parse — don't make the skip path depend on Step 3.
    if "CRYODRGN_META" not in os.environ:
        raise RuntimeError(
            f"{pose_pkl} does not exist and CRYODRGN_META is unset, so there is nothing to "
            f"parse from. Run Step 3 first to point at your .star/.cs.")
    meta = os.environ["CRYODRGN_META"]
    if pose_source == "RELION .star":
        cmd = f'cryodrgn parse_pose_star "{meta}" -o "{pose_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
    else:
        cmd = f'cryodrgn parse_pose_csparc "{meta}" -o "{pose_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
    print("$", cmd, "\n")
    get_ipython().system(cmd)
    _rc = get_ipython().user_ns.get("_exit_code", 0)
    if _rc:
        raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print(f"\n✅ Poses → {pose_pkl}")

In [ ]:
#@title 4.3 · Parse CTF → ctf.pkl (saved to Drive) { display-mode: "form" }
#@markdown Written straight to Drive and skipped on re-run if it already exists. **Re-running
#@markdown this is always safe** — see 4.2. The training cells require the path it exports.
ctf_source = "RELION .star"  #@param ["RELION .star", "cryoSPARC .cs"]
#@markdown For `.star`/`.cs` the box size and Å/px are read from the file; set these only if they
#@markdown are missing (`0` = auto).
box_size_D = 0  #@param {type:"integer"}
apix = 0  #@param {type:"number"}

import os
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
ctf_pkl = os.path.join(DRIVE_DIR, "ctf.pkl")
os.environ["CRYODRGN_CTF"] = ctf_pkl

if os.path.exists(ctf_pkl):
    # As in 4.2: no parse, no overwrite, just exports the path for the training cells.
    print(f"✔ CTF already on Drive — skipping parse: {ctf_pkl}")
    print(f"   ({os.path.getsize(ctf_pkl)/2**20:.1f} MB, last modified "
          f"{__import__('time').ctime(os.path.getmtime(ctf_pkl))})")
else:
    if "CRYODRGN_META" not in os.environ:
        raise RuntimeError(
            f"{ctf_pkl} does not exist and CRYODRGN_META is unset, so there is nothing to "
            f"parse from. Run Step 3 first to point at your .star/.cs.")
    meta = os.environ["CRYODRGN_META"]
    if ctf_source == "RELION .star":
        cmd = f'cryodrgn parse_ctf_star "{meta}" -o "{ctf_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
        if float(apix) > 0:
            cmd += f" --Apix {apix}"
    else:
        cmd = f'cryodrgn parse_ctf_csparc "{meta}" -o "{ctf_pkl}"'
        if int(box_size_D) > 0:
            cmd += f" -D {int(box_size_D)}"
        if float(apix) > 0:
            cmd += f" --Apix {apix}"
    print("$", cmd, "\n")
    get_ipython().system(cmd)
    _rc = get_ipython().user_ns.get("_exit_code", 0)
    if _rc:
        raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print(f"\n✅ CTF → {ctf_pkl}")

In [ ]:
#@title 4.4 · (Optional) Consolidate a chunked stack into one .mrcs { display-mode: "form" }
#@markdown 4.1 writes the downsampled stack in waves (`particles.<D>.0.mrcs`, `.1.mrcs`, …) with
#@markdown a `particles.<D>.txt` listing them. That works, but the multi-file loader rebuilds a
#@markdown pandas `groupby` and a fresh thread pool on **every batch**, and it holds one array
#@markdown per wave during an eager load. Merging the waves into a single `.mrcs` removes both.
#@markdown
#@markdown Measured at box 192, `-b 8`: the per-batch read drops from ~431 µs/particle to
#@markdown ~58 µs, about 175 s → 24 s per 405k-particle epoch. And with a filtering `--ind`
#@markdown the eager load peak halves (2× → 1×), which is often what lets you drop `--lazy`.
#@markdown
#@markdown Particle order is preserved exactly, so **saved `--ind` .pkl files, `pose.pkl` and
#@markdown `ctf.pkl` all stay valid**. Skip this cell if your stack is already a single file.
box_size = 128  #@param [64, 128, 192, 200, 256] {type:"raw"}
#@markdown Folder holding `particles.<D>.txt` and its waves. Blank = wherever Step 4 put the
#@markdown stack, falling back to the Drive project folder.
source_dir = ""  #@param {type:"string"}
#@markdown Where to write `particles.<D>.mrcs`. Blank = the local working folder. **Keep this on
#@markdown local disk** — training reads one particle at a time from it, which is latency-bound
#@markdown and crawls over Drive's FUSE mount.
dest_dir = ""  #@param {type:"string"}
#@markdown Copy the result to Drive too (it is as large as the stack).
copy_to_drive = False  #@param {type:"boolean"}
#@markdown Delete the per-wave .mrcs afterwards, once the output has been verified. Only do this
#@markdown if the waves are also on Drive, or you will have no way back.
remove_waves = False  #@param {type:"boolean"}

import os
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

D = int(box_size)
# Default the source to whatever Step 4 resolved, so this usually needs no fields at all.
_ds = os.environ.get("CRYODRGN_DOWNSAMPLED", "")
src = (source_dir.strip() or (os.path.dirname(_ds) if _ds else "")
       or WORK_DIR or DRIVE_DIR)
if not src or not os.path.isdir(src):
    raise FileNotFoundError(
        "Set source_dir to the folder holding particles.%d.txt and its .mrcs waves." % D)
dst_dir = dest_dir.strip() or WORK_DIR
drive_out = DRIVE_DIR
import shutil, time
import numpy as np
from cryodrgn.mrcfile import MRCHeader

idx = os.path.join(src, "particles.%d.txt" % D)
if not os.path.exists(idx):
    raise FileNotFoundError(
        "%s not found. Point source_dir at the folder holding the chunked stack "
        "(particles.%d.txt plus its particles.%d.N.mrcs waves)." % (idx, D, D))

# The .txt is the authority on ORDER; a glob sort would put .10 before .2 and silently
# permute the stack, invalidating every index file you have.
waves = [ln.strip() for ln in open(idx) if ln.strip()]
waves = [w if os.path.isabs(w) else os.path.join(src, w) for w in waves]
missing = [w for w in waves if not os.path.exists(w)]
if missing:
    raise FileNotFoundError("listed in the index but not on disk: %s" % missing[:3])

hdrs = [MRCHeader.parse(w) for w in waves]
h0 = hdrs[0]
for w, h in zip(waves, hdrs):
    if (h.fields["nx"], h.fields["ny"], h.fields["mode"]) != (
            h0.fields["nx"], h0.fields["ny"], h0.fields["mode"]):
        raise ValueError("%s has nx/ny/mode %s, expected %s" % (
            os.path.basename(w), (h.fields["nx"], h.fields["ny"], h.fields["mode"]),
            (h0.fields["nx"], h0.fields["ny"], h0.fields["mode"])))
if h0.fields["nx"] != D or h0.fields["ny"] != D:
    raise ValueError("the waves are %dx%d, not %d — check box_size."
                     % (h0.fields["ny"], h0.fields["nx"], D))

n_tot = sum(h.fields["nz"] for h in hdrs)
itemsize = np.dtype(h0.dtype).itemsize
payload = n_tot * D * D * itemsize
os.makedirs(dst_dir, exist_ok=True)
out = os.path.join(dst_dir, "particles.%d.mrcs" % D)
free = shutil.disk_usage(dst_dir).free
print("waves     : %d files, %s particles total" % (len(waves), format(n_tot, ",")))
print("output    : %s" % out)
print("size      : %.1f GB (%s dtype)   %.0f GB free" % (
    payload / 1e9, np.dtype(h0.dtype).name, free / 1e9))
print("A/px      : %g" % h0.apix)
if os.path.exists(out):
    raise FileExistsError("%s already exists — delete it or set dest_dir." % out)
if payload > free * 0.95:
    raise RuntimeError("Not enough disk: need %.1f GB, have %.1f GB free.%s"
                       % (payload / 1e9, free / 1e9,
                          "" if dst_dir == src else
                          "  (writing to a different drive than the waves may help)"))

hdr = MRCHeader.make_default_header(nz=n_tot, ny=D, nx=D, dtype=h0.dtype,
                                    is_vol=False, Apix=h0.apix)
t0, done = time.time(), 0
with open(out, "wb") as fo:
    hdr.write(fo)
    for w, h in zip(waves, hdrs):
        start = 1024 + h.fields["next"]            # data begins after any extended header
        want = h.fields["nz"] * D * D * itemsize
        with open(w, "rb") as fi:
            fi.seek(start)
            got = 0
            while got < want:
                buf = fi.read(min(64 << 20, want - got))
                if not buf:
                    raise IOError("%s ended after %d of %d payload bytes"
                                  % (os.path.basename(w), got, want))
                fo.write(buf); got += len(buf)
        done += got
        el = time.time() - t0
        print("  %-26s %6s particles   %5.1f%%  %4.0f MB/s" % (
            os.path.basename(w), format(h.fields["nz"], ","),
            100.0 * done / payload, done / 1e6 / max(el, 1e-9)), flush=True)

got_sz = os.path.getsize(out)
if got_sz != 1024 + payload:
    raise IOError("output is %d bytes, expected %d" % (got_sz, 1024 + payload))
chk = MRCHeader.parse(out)
if chk.fields["nz"] != n_tot or chk.fields["nx"] != D:
    raise IOError("output header reads nz=%d nx=%d" % (chk.fields["nz"], chk.fields["nx"]))
print("\nverified  : %s particles, %dx%d, %g A/px, %.1f GB in %.0f s" % (
    format(chk.fields["nz"], ","), chk.fields["nx"], chk.fields["ny"], chk.apix,
    got_sz / 1e9, time.time() - t0))

if copy_to_drive:
    if not drive_out:
        print("copy_to_drive is on but no Drive folder is known — skipping.")
    else:
        os.makedirs(drive_out, exist_ok=True)
        print("copying to %s ..." % drive_out, flush=True)
        shutil.copyfile(out, os.path.join(drive_out, os.path.basename(out)))
        print("  done")
if remove_waves:
    for w in waves:
        os.remove(w)
    print("removed %d wave file(s); keep particles.%d.txt only if you still want the "
          "chunked view" % (len(waves), D))

print("")
print("Particle order is unchanged — the waves were written in the order this .txt lists")
print("them, which is exactly how TxtFileSource numbered them (source.py:610-623) — so")
print("every saved --ind .pkl, and pose.pkl/ctf.pkl, stay valid with no renumbering.")
print("")
print("On memory: a run WITH a filtering --ind peaks at 1x the array instead of 2x,")
print("because non-contiguous indices take the per-image read path (source.py:425-432)")
print("rather than holding one array per wave file (source.py:511-528). A run WITHOUT")
print("--ind still peaks at 2x either way — the contiguous branch builds the whole stack")
print("before copying it in — so consolidating alone will not rescue an unfiltered run.")
print("The auto lazy decision in the training cells accounts for both.")

# Point the rest of the notebook at the consolidated file. 6.1 and 8.3 read this directly;
# 6.2 takes the stack from the run's config.yaml, so for a RESUME set particles_override
# to the path printed above (or leave it blank if config.yaml's path is already gone, in
# which case 6.2 falls back to this).
os.environ["CRYODRGN_DOWNSAMPLED"] = out
print("")
print("CRYODRGN_DOWNSAMPLED is now %s" % out)
print("  • starting a NEW run (6.1) or a filtered retrain (8.3): nothing else to do.")
print("  • RESUMING a run (6.2): set particles_override to that path — config.yaml still")
print("    names the stack the run started on.")

## 5 · (Optional) Sanity-check poses & CTF

Before spending GPU time on training, back-project a subset of particles into a 3D map.
It should look like a **low-resolution version of your consensus structure**. If it's noise,
the poses/CTF are probably mis-parsed (a common fix is toggling `uninvert_data`).

In [ ]:
#@title 5.1 · Voxel back-projection of a subset { display-mode: "form" }
#@markdown Number of particles to use (fewer = faster, noisier).
n_particles = 10000  #@param {type:"integer"}
#@markdown Tick if your particles are dark-on-light (flips the data sign).
uninvert_data = False  #@param {type:"boolean"}

import os
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
bp_dir = os.path.join(WORK_DIR, "backproject")
os.environ["CRYODRGN_BACKPROJECT"] = bp_dir

cmd = (f'cryodrgn backproject_voxel "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'-o "{bp_dir}" --first {int(n_particles)}')
if uninvert_data:
    cmd += " --uninvert-data"

print("$", cmd, "\n")
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print(f"\n✅ Map → {bp_dir}/backproject.mrc")

In [ ]:
#@title 5.2 · View central slices of the back-projected map { display-mode: "form" }
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn.mrcfile import parse_mrc

bp_dir = os.environ["CRYODRGN_BACKPROJECT"]
hits = glob.glob(os.path.join(bp_dir, "*.mrc"))
if not hits:
    raise FileNotFoundError(f"No .mrc found in {bp_dir} — run cell 5.1 first.")

vol, _ = parse_mrc(hits[0])
D = vol.shape[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (axis, title) in zip(axes, [(0, "Z"), (1, "Y"), (2, "X")]):
    sl = vol.take(D // 2, axis=axis)
    ax.imshow(sl, cmap="Greys_r")
    ax.set_title(f"central {title} slice")
    ax.axis("off")
fig.suptitle(os.path.basename(hits[0]))
plt.tight_layout()
plt.show()
print("Looks like your structure? ✅ Proceed to training.\n"
      "Just noise? ❌ Re-check poses/CTF and try toggling `uninvert_data` in 5.1.")

## 6 · Train the cryoDRGN model

Now train the VAE for heterogeneous reconstruction. Model outputs (per-epoch `weights.*.pkl`,
`z.*.pkl`, `config.yaml`) are written **directly to your Drive project folder**, so they survive a
disconnect. **6.1** starts a fresh run (and refuses to overwrite an existing one); **6.2** resumes
or extends a run from its latest checkpoint — even in a brand-new session, once Steps 2–4 have been
re-run (4.1 restores the stack from Drive, 4.2/4.3 skip).

**Tips:** `zdim` 8 is a good default (use 1 for a single motion axis, ≥10 for complex mixtures).
For a first pass, 25 epochs at `D=128` is typical. On a T4, `D=128` runs roughly a few minutes/epoch
for ~100k particles; `D=256` is far slower.

**Lazy loading is off unless it has to be on.** By default cryoDRGN holds the whole (post-`--ind`)
stack in RAM as float32 — `n × D² × 4` bytes — which is much faster than re-reading images from
Drive every epoch. `--lazy` trades that speed for memory. The `lazy` field in 6.1 / 6.2 / 8.3
defaults to `auto`: each cell measures what the array will cost against what the runtime actually
has free and only passes `--lazy` when it will not fit. Set it to `yes`/`no` to override.

| box `D` | 405k particles, eager | verdict on a 51 GB High-RAM runtime |
|---|---|---|
| 128 | 26.6 GB | fits — no `--lazy` |
| 256 | 106.2 GB | does not fit — `--lazy` required |

**AMP is on by default and the notebook leaves it that way.** `--no-amp` is
`action="store_false", dest="amp"` (`train_vae.py:245-250`), so `args.amp` starts `True`. It logs
a warning for any of batch size, `D-1`, `pdim`, `qdim`, `zdim` or the masked input dimension not
divisible by 8 — those are speed hints, not errors. Note `save_checkpoint` does not persist the
`GradScaler`, so a resume restarts its loss scale; it re-converges within a few steps.

In [ ]:
#@title 6.1 · Configure & launch train_vae { display-mode: "form" }
#@markdown **Latent dimension** — size of the conformational latent space.
zdim = 8  #@param [1, 2, 4, 8, 10] {type:"raw"}
#@markdown **Epochs** — full passes over the dataset.
num_epochs = 25  #@param {type:"integer"}
#@markdown **Batch size** — increase to better use a big GPU (affects dynamics).
#@markdown Minibatch size. **16 is cryoDRGN's own default** — changing it changes the
#@markdown number of Adam steps per epoch at a fixed learning rate, so keep it constant
#@markdown across a run and record it if you compare models.
batch_size = 16  #@param [8, 16, 32] {type:"raw"}
#@markdown **Output folder name** (created inside your Drive project folder).
output_name = "00_cryodrgn128"  #@param {type:"string"}
#@markdown Dark-on-light particles? (must match what worked in Step 5)
uninvert_data = False  #@param {type:"boolean"}
#@markdown Lazy loading. **`auto`** keeps it off — which is what you want, it is much faster —
#@markdown and only switches it on when the whole stack will not fit in RAM as float32.
lazy = "auto"  #@param ["auto", "no", "yes"]
#@markdown RAM to leave free for everything that is not the particle array (CUDA context,
#@markdown workers, the notebook). Only used by `auto`.
ram_headroom_gib = 4  #@param {type:"number"}
#@markdown DataLoader worker processes. **`-1` = auto**, `0` = load in the main process,
#@markdown `N` = that many. Auto picks 0 on a GPU runtime, because train_vae puts the dataset
#@markdown on the GPU, so workers hand CUDA tensors back over IPC — a per-batch cost that adding
#@markdown workers does not reduce. On a CPU-only run auto uses `min(8, cores-2)`.
num_workers = -1  #@param {type:"integer"}
#@markdown Tick only to **start over** in a folder that already has a run (otherwise the cell stops
#@markdown and sends you to 6.2 to continue it — this guards against overwriting on a "Run all").
overwrite = False  #@param {type:"boolean"}

import os, glob
from cryodrgn.source import ImageSource
ds = os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
outdir = os.path.join(DRIVE_DIR, output_name)
os.environ["CRYODRGN_OUTDIR"] = outdir

if glob.glob(os.path.join(outdir, "weights.*.pkl")) and not overwrite:
    raise FileExistsError(
        f"{outdir} already contains checkpoints. To CONTINUE this run, use cell 6.2 (resume). "
        f"To START OVER, tick `overwrite` above or change `output_name`.")

_n_load = ImageSource.from_file(ds, lazy=True).n   # 6.1 trains on the whole stack
_ind_arr = None                                    # ...so there is no --ind to exploit
# --- lazy loading: on only when the eager array will not fit -------------------------
# ImageDataset materializes the SELECTED particles as float32 (source.py:116-126), so the
# steady-state cost is n * D^2 * 4 bytes. The PEAK during the load is what actually decides
# whether the runtime survives, and it is 2x that in every case but one -- measured, not
# assumed:
#   .mrcs, no/contiguous --ind  2.00x   contiguous branch reads the whole stack with one
#                                       np.fromfile and copies it into the preallocated
#                                       array, so both exist at once (source.py:419-422)
#   .txt,  no/contiguous --ind  2.01x   one future per member file, all held in `to_do`
#                                       until the read finishes (source.py:511-528)
#   .txt,  filtering --ind      2.01x   same -- each future still materializes its file's
#                                       whole share, so their sum is the full stack
#   .mrcs, filtering --ind      1.00x   non-contiguous indices take the per-image loop
#                                       (source.py:425-432): one image in flight at a time
# So it is NOT "single file is cheaper" in general -- a plain full load costs 2x either way.
# The saving is specific to a single .mrcs read through a non-contiguous --ind, which is
# exactly the filtered-retraining case. Lazy avoids the peak entirely, but then re-reads
# every image from disk every epoch, which on a Drive-backed stack is far slower. Hence:
# off unless it must be on.
import psutil
import numpy as np
from cryodrgn.source import ImageSource
_D = ImageSource.from_file(ds, lazy=True).D
_single = os.path.splitext(ds)[1].lower() in (".mrc", ".mrcs")
_contig = True
try:
    _ia = np.asarray(_ind_arr).ravel() if _ind_arr is not None else None
    if _ia is not None and _ia.size and not np.issubdtype(_ia.dtype, np.bool_):
        _contig = bool(np.all(_ia == _ia[0] + np.arange(_ia.size)))
except Exception:
    _contig = True                      # unsure -> assume the expensive path
_mult = 1 if (_single and not _contig) else 2
_need = _n_load * _D * _D * 4 * _mult
_avail = psutil.virtual_memory().available
_budget = _avail - float(ram_headroom_gib) * 2**30
use_lazy = (_need > _budget) if lazy == "auto" else (lazy == "yes")

_why = ("  (x2: multi-file loader holds every per-file array)" if _mult == 2 and not _single
        else "  (x2: contiguous read builds the whole stack before copying it)"
        if _mult == 2 else "  (x1: single file + non-contiguous --ind reads one image at a time)")
print(f"eager RAM : {_need / 2**30:.1f} GiB for {_n_load:,} x {_D}^2 float32" + _why)
print(f"            {_avail / 2**30:.1f} GiB free now, {_budget / 2**30:.1f} GiB budget "
      f"after {ram_headroom_gib} GiB headroom")
print(f"lazy      : {'ON' if use_lazy else 'off'}  "
      + (("(auto: the stack does not fit)" if use_lazy else "(auto: the stack fits)")
         if lazy == "auto" else "(set explicitly)"))
if use_lazy:
    print("            lazy re-reads every image each epoch — expect slower epochs. A")
    print("            High-RAM runtime, or a smaller box, would let you keep it off.")
    # The one case with a cheap fix: a chunked .txt read through a filtering --ind pays 2x
    # only because of the per-file futures. The same particles in one .mrcs take the
    # per-image path instead and cost 1x — which here would be enough to stay eager.
    if not _single and not _contig and (_need // 2) <= _budget:
        print(f"            💡 this stack is {os.path.splitext(ds)[1]} (multi-file). Consolidating it into a")
        print(f"            single .mrcs — cell 6.1 of the preprocess notebook — drops the load")
        print(f"            peak to {_need / 2 / 2**30:.1f} GiB, which fits, so this run could stay eager.")
        print(f"            Particle order and every index are preserved, so --ind/poses/ctf")
        print(f"            carry over unchanged.")
elif lazy == "auto" and _need > _budget * 0.8:
    print(f"            ⚠️  only {(_budget - _need) / 2**30:.1f} GiB spare. If the runtime dies")
    print("            with no traceback, that was the OOM killer — set lazy='yes'.")

# With --lazy every image is re-read and re-transformed every epoch, and train_vae gives that
# job just TWO worker processes (train_vae.py:874-880). On a multi-core runtime that, not the
# GPU, is what sets the step time. Each worker also fans out over max_threads for multi-file
# sources, so leave headroom rather than claiming every core.
_ncpu = os.cpu_count() or 2
try:
    import torch as _t
    _cuda = _t.cuda.is_available()
except Exception:
    _cuda = False

# On a GPU runtime train_vae builds ImageDataset with device='cuda' (train_vae.py:693), and
# __getitem__ does .to(self.device) BEFORE _process (dataset.py:128) -- so the transform runs
# on the GPU and every worker returns a CUDA tensor. Handing those across a process boundary
# goes through CUDA IPC, which PyTorch's own docs advise against; it costs an IPC handle per
# batch and a CUDA context per worker, and it is what emits
#   "Producer process has been terminated before all shared CUDA tensors released".
# Measured on an L4 at box 192: 2 workers and 8 workers both give ~17 s/1000 particles, so
# the extra processes buy nothing while adding that per-batch cost. Default to 0 on a GPU.
# -1 = auto; 0 means literally zero, matching DataLoader's own num_workers semantics.
# (An earlier version overloaded 0 as "auto", which read as "no workers" and silently
# gave 8 of them -- exactly the case this whole note is about.)
_nw = int(num_workers)
if _nw >= 0:
    n_workers = _nw
    print(f"workers   : {n_workers} (set explicitly) of {_ncpu} CPU(s)"
          + ("  — main process loads, no IPC" if n_workers == 0 else ""))
    if n_workers and _cuda:
        print("            ⚠️  the dataset is CUDA-resident, so workers return CUDA tensors")
        print("            over IPC — a per-batch cost that does not shrink as you add more.")
        print("            If steps are slow, set num_workers = 0.")
elif _cuda:
    n_workers = 0
    print(f"workers   : 0 (auto — the dataset lives on the GPU, so extra processes would")
    print(f"            only add CUDA IPC per batch; {_ncpu} CPU(s) available if you want to")
    print(f"            override)")
elif use_lazy:
    n_workers = min(8, max(2, _ncpu - 2))
    print(f"workers   : {n_workers} of {_ncpu} CPU(s)  (auto — CPU-only run, so the dataset "
          f"is CPU-resident and workers genuinely parallelise the load)")
else:
    n_workers = 0     # eager on CPU: cryoDRGN loads in the main process

if not use_lazy and n_workers:
    # Workers fork, so the resident array is shared copy-on-write and is NOT duplicated --
    # images() hands out a fancy-indexed copy (source.py:256), so _process's in-place ops
    # never write back to it. What each worker does cost is its own interpreter and
    # buffers, and in eager mode that comes out of the budget the array has already nearly
    # filled. They are still worth it: _process is the only CPU work left once the stack is
    # resident, and cryoDRGN leaves it single-threaded here.
    _wcost = n_workers * 0.6 * 2**30
    _left = _budget - _need - _wcost
    print(f"            + ~{_wcost / 2**30:.1f} GiB for {n_workers} eager worker(s) "
          f"-> {_left / 2**30:.1f} GiB would remain")
    if _left < 0:
        raise MemoryError(
            f"{n_workers} eager workers need ~{_wcost / 2**30:.1f} GiB on top of the "
            f"{_need / 2**30:.1f} GiB array, which overruns the "
            f"{_budget / 2**30:.1f} GiB budget.\n  Lower num_workers, or set it to 0 to let "
            f"the main process load as cryoDRGN does by default.")

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--zdim {zdim} -n {int(num_epochs)} -b {batch_size} -o "{outdir}"')
if uninvert_data:
    cmd += " --uninvert-data"
if use_lazy:
    cmd += " --lazy"
cmd += f" --num-workers {n_workers}"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print("=" * 70 + f"\n✅ Training complete. Model saved to: {outdir}")

In [ ]:
#@title 6.2 · Resume / extend a run — any session { display-mode: "form" }
#@markdown Continue an existing model, **including after a disconnect** — you do NOT need to have
#@markdown run 6.1 this session. Just re-run Steps 2–4 first (4.1 restores the stack from Drive,
#@markdown 4.2/4.3 skip), then point this at the run's folder. It picks up from the latest saved
#@markdown checkpoint automatically (`--load latest`) and trains up to `num_epochs` total.
#@markdown
#@markdown **This is also how you resume a filtered run from 8.3.** Everything that defines the
#@markdown dataset — the stack, poses, CTF, `--ind`, and the preprocessing flags — is rebuilt from
#@markdown that run's `config.yaml`, so the subset is preserved. Do **not** re-run 8.3 (it starts
#@markdown a fresh model and refuses a folder that already has checkpoints) or 6.1 (it has no
#@markdown `--ind`, so it would train on the whole stack).
output_name = "00_cryodrgn128"  #@param {type:"string"}
#@markdown Total epochs to reach (must exceed the last completed epoch).
num_epochs = 25  #@param {type:"integer"}
#@markdown `train_vae` runs `analyze` on the final epoch automatically (~4 min). Untick when
#@markdown stepping up in stages (10→15→20→…) to skip it on the intermediate runs; every epoch
#@markdown still saves `z.N.pkl`/`weights.N.pkl`, so you can analyze any of them later.
run_analysis = True  #@param {type:"boolean"}
#@markdown Minibatch size — **`0` inherits the previous session's value from `run.log`**. Keep it
#@markdown constant: halving `-b` doubles the number of Adam steps per epoch, so "epoch" stops
#@markdown being a uniform unit and every convergence plot's x-axis is silently stretched.
batch_size = 0  #@param [0, 8, 16, 32] {type:"raw"}
#@markdown Repackaged stack to resume against — blank keeps the one in `config.yaml`. Use this
#@markdown **only** for a byte-identical repackaging of the same data, e.g. swapping a chunked
#@markdown `particles.<D>.txt` for the single `particles.<D>.mrcs` that preprocess cell 6.1
#@markdown builds. With a filtering `--ind` that halves the eager peak and can turn `--lazy`
#@markdown off. The particle count is checked against the original before anything runs.
#@markdown **Put it on local disk, not Drive** — the 1x path issues one read per particle,
#@markdown which is latency-bound and crawls over Drive's FUSE mount.
particles_override = ""  #@param {type:"string"}
#@markdown Lazy loading. `save_config` does not record it, so it cannot be inherited from
#@markdown `config.yaml` — but unlike `-b` it only changes I/O, never the optimisation, so it is
#@markdown safe to redecide each session. **`auto`** keeps it off unless the stack will not fit.
lazy = "auto"  #@param ["auto", "no", "yes"]
#@markdown RAM to leave free for everything that is not the particle array. Only used by `auto`.
ram_headroom_gib = 4  #@param {type:"number"}
#@markdown DataLoader workers. **`-1` = auto** (0 on a GPU runtime — see 6.1), `0` = main
#@markdown process, `N` = that many. Eager workers also cost ~0.6 GiB each.
num_workers = -1  #@param {type:"integer"}

import os, re, glob
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
outdir = os.path.join(DRIVE_DIR, output_name)

if not os.path.exists(os.path.join(outdir, "config.yaml")):
    raise FileNotFoundError(
        f"No run found at {outdir} — check output_name (it must match the folder 6.1 or 8.3 "
        f"wrote to).")

done = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "weights.*.pkl"))
        for m in [re.search(r"weights\.(\d+)\.pkl$", os.path.basename(p))] if m]
if not done:
    raise FileNotFoundError(f"No numbered checkpoints in {outdir} — nothing to resume from.")
last = max(done)
if int(num_epochs) <= last:
    raise ValueError(f"num_epochs ({num_epochs}) must exceed the last checkpoint (epoch {last}).")

# Every argument that defines the model or the dataset must be reproduced exactly, or the
# resumed run silently differs from the original (or fails to load the weights). config.yaml
# records them all, so rebuild the command from it rather than from the form fields.
import yaml
cfg = yaml.safe_load(open(os.path.join(outdir, "config.yaml")))
m, da = cfg["model_args"], cfg["dataset_args"]
zdim = m["zdim"]

# The stack, poses and CTF come from config.yaml, NOT from the environment. 8.3 lets you
# train on a specific file (e.g. a single wave, particles.128.0.mrcs) while cell 4/5 leave
# CRYODRGN_DOWNSAMPLED pointing at the whole stack (particles.128.txt when --chunk was used).
# Resuming off the env var would silently continue the run on a DIFFERENT dataset -- the
# --ind indices would then select different particles and nothing would raise.
def _from_cfg(key, env, label):
    val = da.get(key)
    if val and os.path.exists(val):
        return val
    fallback = os.environ.get(env)
    if val:
        print(f"⚠️  config.yaml records {label} = {val}\n    but that path is gone; "
              f"falling back to {fallback}. Check this is the same data.")
    if not fallback:
        raise FileNotFoundError(
            f"config.yaml has no usable {label} and {env} is unset — re-run Step 4 first.")
    return fallback

if particles_override.strip():
    # Resolve the override FIRST and do not require the run's own `particles` path to still
    # exist. That path is typically a /content/... working file, so on a fresh runtime it is
    # gone -- which is precisely when you reach for this field. Making the override depend on
    # _from_cfg would block the case it exists for.
    _new = os.path.abspath(particles_override.strip())
    if not os.path.exists(_new):
        raise FileNotFoundError(f"particles_override: {_new} not found.")
    # Swapping the stack under a resume is only safe when the new file holds the SAME images
    # in the SAME order -- a repackaging, not a different dataset. The particle count is the
    # one cheap invariant that catches a genuine mix-up, so require it to match whenever
    # anything is still around to compare against.
    _n_new = ImageSource.from_file(_new, lazy=True).n
    _cfg_ds = da.get("particles")
    _ref = _cfg_ds if (_cfg_ds and os.path.exists(_cfg_ds)) else os.environ.get(
        "CRYODRGN_DOWNSAMPLED")
    _n_old = (ImageSource.from_file(_ref, lazy=True).n
              if (_ref and os.path.exists(_ref)) else None)
    if _n_old is not None and _n_new != _n_old:
        raise ValueError(
            f"particles_override holds {_n_new:,} particles but the stack this run was "
            f"trained on ({os.path.basename(_ref)}) holds {_n_old:,}.\n  These are different "
            f"datasets — --ind and the learned weights would no longer line up. Refusing.")
    if _n_old is None:
        print(f"⚠️  neither {_cfg_ds} nor CRYODRGN_DOWNSAMPLED is reachable, so the particle "
              f"count could not be cross-checked.\n    Verify {_n_new:,} is the whole stack "
              f"(NOT the filtered subset) before trusting this run.")
    print(f"override  : {os.path.basename(_cfg_ds or '?')} -> {os.path.basename(_new)}  "
          f"({_n_new:,} particles, unchanged)")
    ds = _new
else:
    ds = _from_cfg("particles", "CRYODRGN_DOWNSAMPLED", "particles")
pose = _from_cfg("poses", "CRYODRGN_POSE", "poses")
ctf = _from_cfg("ctf", "CRYODRGN_CTF", "ctf")
print(f"stack     : {ds}")
_env_ds = os.environ.get("CRYODRGN_DOWNSAMPLED")
if _env_ds and ds != _env_ds:
    print(f"            (this run was trained on a specific file, not the "
          f"{os.path.basename(_env_ds)} that Step 4 resolved)")

arch = (f'--zdim {zdim} --enc-dim {m["qdim"]} --enc-layers {m["qlayers"]} '
        f'--dec-dim {m["pdim"]} --dec-layers {m["players"]}')
extra = ""
if da.get("invert_data") is False:      # --uninvert-data was used; omitting it flips the sign
    extra += " --uninvert-data"
if da.get("ind"):                       # trained on a subset; omitting it changes the dataset
    extra += f' --ind "{da["ind"]}"'
if da.get("window") is False:
    extra += " --no-window"
if da.get("window_r") not in (None, 0.85):
    extra += f' --window-r {da["window_r"]}'
if da.get("datadir"):
    extra += f' --datadir "{da["datadir"]}"'

os.environ["CRYODRGN_OUTDIR"] = outdir  # so Steps 7-9 target this run
print("carried over from config.yaml:", arch + (extra or " (no extra flags)"))

# how many particles will actually be resident? --ind restricts it: ImageDataset forwards it
# as `indices=` to ImageSource (dataset.py:49-55) and the eager path materializes only those.
_ind = da.get("ind")
if _ind and os.path.exists(_ind):
    _sel = np.asarray(utils.load_pkl(_ind)).ravel()
    _n_load = int(_sel.sum()) if _sel.dtype == bool else int(_sel.size)
    _ind_arr = _sel
    print(f"--ind     : {_n_load:,} of {ImageSource.from_file(ds, lazy=True).n:,} particles")
else:
    if _ind:
        print(f"⚠️  config.yaml records --ind {_ind} but that file is missing — the resume "
              f"will fail. Restore it, or the run's dataset silently changes.")
    _n_load = ImageSource.from_file(ds, lazy=True).n
    _ind_arr = None
# --- lazy loading: on only when the eager array will not fit -------------------------
# ImageDataset materializes the SELECTED particles as float32 (source.py:116-126), so the
# steady-state cost is n * D^2 * 4 bytes. The PEAK during the load is what actually decides
# whether the runtime survives, and it is 2x that in every case but one -- measured, not
# assumed:
#   .mrcs, no/contiguous --ind  2.00x   contiguous branch reads the whole stack with one
#                                       np.fromfile and copies it into the preallocated
#                                       array, so both exist at once (source.py:419-422)
#   .txt,  no/contiguous --ind  2.01x   one future per member file, all held in `to_do`
#                                       until the read finishes (source.py:511-528)
#   .txt,  filtering --ind      2.01x   same -- each future still materializes its file's
#                                       whole share, so their sum is the full stack
#   .mrcs, filtering --ind      1.00x   non-contiguous indices take the per-image loop
#                                       (source.py:425-432): one image in flight at a time
# So it is NOT "single file is cheaper" in general -- a plain full load costs 2x either way.
# The saving is specific to a single .mrcs read through a non-contiguous --ind, which is
# exactly the filtered-retraining case. Lazy avoids the peak entirely, but then re-reads
# every image from disk every epoch, which on a Drive-backed stack is far slower. Hence:
# off unless it must be on.
import psutil
import numpy as np
from cryodrgn.source import ImageSource
_D = ImageSource.from_file(ds, lazy=True).D
_single = os.path.splitext(ds)[1].lower() in (".mrc", ".mrcs")
_contig = True
try:
    _ia = np.asarray(_ind_arr).ravel() if _ind_arr is not None else None
    if _ia is not None and _ia.size and not np.issubdtype(_ia.dtype, np.bool_):
        _contig = bool(np.all(_ia == _ia[0] + np.arange(_ia.size)))
except Exception:
    _contig = True                      # unsure -> assume the expensive path
_mult = 1 if (_single and not _contig) else 2
_need = _n_load * _D * _D * 4 * _mult
_avail = psutil.virtual_memory().available
_budget = _avail - float(ram_headroom_gib) * 2**30
use_lazy = (_need > _budget) if lazy == "auto" else (lazy == "yes")

_why = ("  (x2: multi-file loader holds every per-file array)" if _mult == 2 and not _single
        else "  (x2: contiguous read builds the whole stack before copying it)"
        if _mult == 2 else "  (x1: single file + non-contiguous --ind reads one image at a time)")
print(f"eager RAM : {_need / 2**30:.1f} GiB for {_n_load:,} x {_D}^2 float32" + _why)
print(f"            {_avail / 2**30:.1f} GiB free now, {_budget / 2**30:.1f} GiB budget "
      f"after {ram_headroom_gib} GiB headroom")
print(f"lazy      : {'ON' if use_lazy else 'off'}  "
      + (("(auto: the stack does not fit)" if use_lazy else "(auto: the stack fits)")
         if lazy == "auto" else "(set explicitly)"))
if use_lazy:
    print("            lazy re-reads every image each epoch — expect slower epochs. A")
    print("            High-RAM runtime, or a smaller box, would let you keep it off.")
    # The one case with a cheap fix: a chunked .txt read through a filtering --ind pays 2x
    # only because of the per-file futures. The same particles in one .mrcs take the
    # per-image path instead and cost 1x — which here would be enough to stay eager.
    if not _single and not _contig and (_need // 2) <= _budget:
        print(f"            💡 this stack is {os.path.splitext(ds)[1]} (multi-file). Consolidating it into a")
        print(f"            single .mrcs — cell 6.1 of the preprocess notebook — drops the load")
        print(f"            peak to {_need / 2 / 2**30:.1f} GiB, which fits, so this run could stay eager.")
        print(f"            Particle order and every index are preserved, so --ind/poses/ctf")
        print(f"            carry over unchanged.")
elif lazy == "auto" and _need > _budget * 0.8:
    print(f"            ⚠️  only {(_budget - _need) / 2**30:.1f} GiB spare. If the runtime dies")
    print("            with no traceback, that was the OOM killer — set lazy='yes'.")

# With --lazy every image is re-read and re-transformed every epoch, and train_vae gives that
# job just TWO worker processes (train_vae.py:874-880). On a multi-core runtime that, not the
# GPU, is what sets the step time. Each worker also fans out over max_threads for multi-file
# sources, so leave headroom rather than claiming every core.
_ncpu = os.cpu_count() or 2
try:
    import torch as _t
    _cuda = _t.cuda.is_available()
except Exception:
    _cuda = False

# On a GPU runtime train_vae builds ImageDataset with device='cuda' (train_vae.py:693), and
# __getitem__ does .to(self.device) BEFORE _process (dataset.py:128) -- so the transform runs
# on the GPU and every worker returns a CUDA tensor. Handing those across a process boundary
# goes through CUDA IPC, which PyTorch's own docs advise against; it costs an IPC handle per
# batch and a CUDA context per worker, and it is what emits
#   "Producer process has been terminated before all shared CUDA tensors released".
# Measured on an L4 at box 192: 2 workers and 8 workers both give ~17 s/1000 particles, so
# the extra processes buy nothing while adding that per-batch cost. Default to 0 on a GPU.
# -1 = auto; 0 means literally zero, matching DataLoader's own num_workers semantics.
# (An earlier version overloaded 0 as "auto", which read as "no workers" and silently
# gave 8 of them -- exactly the case this whole note is about.)
_nw = int(num_workers)
if _nw >= 0:
    n_workers = _nw
    print(f"workers   : {n_workers} (set explicitly) of {_ncpu} CPU(s)"
          + ("  — main process loads, no IPC" if n_workers == 0 else ""))
    if n_workers and _cuda:
        print("            ⚠️  the dataset is CUDA-resident, so workers return CUDA tensors")
        print("            over IPC — a per-batch cost that does not shrink as you add more.")
        print("            If steps are slow, set num_workers = 0.")
elif _cuda:
    n_workers = 0
    print(f"workers   : 0 (auto — the dataset lives on the GPU, so extra processes would")
    print(f"            only add CUDA IPC per batch; {_ncpu} CPU(s) available if you want to")
    print(f"            override)")
elif use_lazy:
    n_workers = min(8, max(2, _ncpu - 2))
    print(f"workers   : {n_workers} of {_ncpu} CPU(s)  (auto — CPU-only run, so the dataset "
          f"is CPU-resident and workers genuinely parallelise the load)")
else:
    n_workers = 0     # eager on CPU: cryoDRGN loads in the main process

if not use_lazy and n_workers:
    # Workers fork, so the resident array is shared copy-on-write and is NOT duplicated --
    # images() hands out a fancy-indexed copy (source.py:256), so _process's in-place ops
    # never write back to it. What each worker does cost is its own interpreter and
    # buffers, and in eager mode that comes out of the budget the array has already nearly
    # filled. They are still worth it: _process is the only CPU work left once the stack is
    # resident, and cryoDRGN leaves it single-threaded here.
    _wcost = n_workers * 0.6 * 2**30
    _left = _budget - _need - _wcost
    print(f"            + ~{_wcost / 2**30:.1f} GiB for {n_workers} eager worker(s) "
          f"-> {_left / 2**30:.1f} GiB would remain")
    if _left < 0:
        raise MemoryError(
            f"{n_workers} eager workers need ~{_wcost / 2**30:.1f} GiB on top of the "
            f"{_need / 2**30:.1f} GiB array, which overruns the "
            f"{_budget / 2**30:.1f} GiB budget.\n  Lower num_workers, or set it to 0 to let "
            f"the main process load as cryoDRGN does by default.")

# save_config stores dataset/lattice/model args only — no batch size, lr or epochs — so
# config.yaml cannot tell us what -b the run has been using. train_vae does log its full
# Namespace to run.log at every session start (train_vae.py:641), which records the EFFECTIVE
# value including argparse defaults, so recover it from there. Omitting -b silently falls back
# to train_vae's default of 16 regardless of what 6.1 used.
bs = int(batch_size)
if bs <= 0:
    seen = []
    logp = os.path.join(outdir, "run.log")
    if os.path.exists(logp):
        with open(logp) as f:
            for line in f:
                if "Namespace(" in line:
                    mb = re.search(r"batch_size=(\d+)", line)
                    if mb:
                        seen.append(int(mb.group(1)))
    if seen:
        bs = seen[-1]
        if len(set(seen)) > 1:
            print(f"⚠️  batch size has already varied across sessions: {seen}")
            print("    'epoch' is therefore not a uniform unit of optimisation in this run —")
            print("    later epochs contain proportionally more Adam steps. Keep it fixed now.")
        print(f"inheriting -b {bs} from the last session recorded in run.log")
    else:
        bs = 16
        print(f"no Namespace line in run.log — falling back to train_vae's default -b {bs}")

# run.log's Namespace also records whether the previous session used --lazy. save_config
# does not, so the decision is remade each time; if it comes out different from last time
# that is worth surfacing -- usually it means a smaller runtime, or a stack that changed
# shape (single .mrcs vs chunked .txt, which doubles the eager peak).
_prev_lazy = None
_logp = os.path.join(outdir, "run.log")
if os.path.exists(_logp):
    for line in open(_logp):
        if "Namespace(" in line:
            _m = re.search(r"lazy=(True|False)", line)
            if _m:
                _prev_lazy = _m.group(1) == "True"
if _prev_lazy is not None and _prev_lazy != use_lazy:
    print(f"⚠️  the previous session ran with lazy={_prev_lazy}, this one will use "
          f"lazy={use_lazy}.")
    print("    That is allowed -- lazy changes only I/O, never the optimisation -- but if it")
    print("    flipped OFF unexpectedly, check the RAM line above before launching: an eager")
    print("    load that does not fit is killed by the OOM killer with no traceback.")

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'{arch}{extra} -n {int(num_epochs)} -b {bs} -o "{outdir}" --load latest')
if use_lazy:
    cmd += " --lazy"
cmd += f" --num-workers {n_workers}"
if not run_analysis:
    cmd += " --no-analysis"
print(f"Resuming '{output_name}' from epoch {last} → {int(num_epochs)}")
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print("=" * 70 + "\n✅ Done. Re-run Step 7 (7.1) to analyze the extended model.")

## 7 · Analyze the results

`cryodrgn analyze` visualizes the latent space (PCA + UMAP), then generates representative
volumes by k-means-sampling the latent space and traversing its principal components.

> **`train_vae` already ran this for you.** It calls `analyze` automatically on the final
> epoch (unless `--no-analysis`), so `analyze.<final-epoch>/` exists as soon as 6.1 or 6.2
> finishes — that is what the `Running UMAP...` lines at the end of the training log were.
> Use 7.1 only to analyze a **different** epoch, or to redo it with different `--ksample`
> / `--Apix`; skip straight to 7.2 to view the plots that already exist.

In [ ]:
#@title 7.1 · Run cryodrgn analyze { display-mode: "form" }
#@markdown Epoch to analyze — leave at **-1** to auto-pick the latest saved epoch.
epoch = -1  #@param {type:"integer"}
#@markdown Number of k-means volumes to generate.
ksample = 20  #@param {type:"integer"}
#@markdown Pixel size (Å/px) written into volume headers (`0` = read from ctf.pkl / default 1).
apix = 0  #@param {type:"number"}

import os, re, glob
outdir = os.environ["CRYODRGN_OUTDIR"]

if int(epoch) < 0:  # auto-detect latest z.N.pkl
    epochs = []
    for p in glob.glob(os.path.join(outdir, "z.*.pkl")):
        m = re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))
        if m:
            epochs.append(int(m.group(1)))
    if not epochs:
        raise FileNotFoundError(f"No z.N.pkl checkpoints in {outdir} — has 6.1 finished?")
    epoch = max(epochs)
    print(f"Auto-selected latest epoch: {epoch}")

os.environ["CRYODRGN_EPOCH"] = str(int(epoch))
cmd = f'cryodrgn analyze "{outdir}" {int(epoch)} --ksample {int(ksample)}'
if float(apix) > 0:
    cmd += f" --Apix {apix}"

print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print("=" * 70 + f"\n✅ Analysis → {outdir}/analyze.{int(epoch)}")

In [ ]:
#@title 7.2 · View latent-space plots inline { display-mode: "form" }
import os, re, glob
from IPython.display import Image, display, Markdown

outdir = os.environ["CRYODRGN_OUTDIR"]
# 7.1 sets CRYODRGN_EPOCH, but train_vae already ran analyze for you, so 7.1 is often
# skipped — and in a fresh session it was never run at all. Fall back to the newest folder.
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
adir = os.path.join(outdir, f"analyze.{epoch}")

for fname, caption in [
    ("z_pca.png", "**PCA** of the latent embeddings (colored by k-means cluster)"),
    ("umap.png", "**UMAP** of the latent embeddings"),
    ("z_pca_marginals.png", "PCA with marginal distributions"),
    ("umap_marginals.png", "UMAP with marginal distributions"),
    (f"learning_curve_epoch{epoch}.png", "Training loss curve"),
]:
    path = os.path.join(adir, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(path, width=520))
    else:
        # k-means subfolder holds some variants
        alt = glob.glob(os.path.join(adir, "kmeans*", fname))
        if alt:
            display(Markdown(caption))
            display(Image(alt[0], width=520))

In [ ]:
#@title 7.3 · Interactive 3D view of a generated volume { display-mode: "form" }
#@markdown Renders one of the k-means representative maps as a 3D isosurface you can rotate.
#@markdown Change `volume_index` (0 … ksample-1) to inspect different structures.
volume_index = 0  #@param {type:"integer"}
#@markdown Isosurface threshold as a percentile of density (higher = tighter surface).
iso_percentile = 99.0  #@param {type:"slider", min:90, max:99.9, step:0.1}
#@markdown Downsample the box for a snappier render.
display_box = 64  #@param [48, 64, 96] {type:"raw"}

import os, re, glob
import numpy as np
import plotly.graph_objects as go
from cryodrgn.mrcfile import parse_mrc

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:   # 7.1 skipped, or a fresh session — take the newest analyze.N/
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
adir = os.path.join(outdir, f"analyze.{epoch}")

vols = sorted(glob.glob(os.path.join(adir, "kmeans*", "vol_*.mrc")))
if not vols:
    raise FileNotFoundError(f"No k-means volumes in {adir} — run 7.1 without --skip-vol.")
vol_path = vols[int(volume_index) % len(vols)]
vol, _ = parse_mrc(vol_path)

# light box downsampling for display
D = vol.shape[0]
if D > int(display_box):
    step = int(round(D / int(display_box)))
    vol = vol[::step, ::step, ::step]
D = vol.shape[0]

x, y, z = np.mgrid[0:D, 0:D, 0:D]
iso = float(np.percentile(vol, iso_percentile))
fig = go.Figure(go.Isosurface(
    x=x.flatten(), y=y.flatten(), z=z.flatten(), value=vol.flatten(),
    isomin=iso, isomax=float(vol.max()),
    surface_count=1, colorscale="Greys", showscale=False, caps=dict(x_show=False, y_show=False, z_show=False),
))
fig.update_layout(title=os.path.basename(vol_path), width=560, height=560,
                  scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
fig.show()
print(f"Showing {vol_path}  ({len(vols)} volumes available; set volume_index 0..{len(vols)-1})")

### 7.4–7.6 · Convergence & overfitting diagnostics

**7.4** parses `run.log` and plots total / reconstruction / KLD loss per epoch. It is cheap and
reads the log *in place*, so you can run it **while training is still going** to watch progress.

> **Diagnosing a finished run in a fresh session.** All three cells take a `model_folder` field, so
> they don't need Step 6 to have run. Because the run folder lives in your Drive project, the only
> prerequisites are **1.1–1.3** (install) and **2.1–2.2** (mount + project folder) — then type the
> run name (e.g. `00_cryodrgn128`) into 7.5 and go. **Steps 3, 4, 5, 6 and 7.1 are all skippable:**
> nothing in 7.4–7.6 loads the particle stack or the poses, and even `full` mode decodes its
> volumes from `weights.N.pkl` + `config.yaml`. (7.5 does read `ctf.pkl` for the pixel size, but
> straight from the path stored in `config.yaml`, and falls back to 1.0 Å/px if it has moved.)

**7.5** runs `cryodrgn_utils analyze_convergence`, which goes well beyond the loss curve:

| | Metric | Reads |
|---|---|---|
| 1 | Total loss curve | `run.log` |
| 2 | UMAP of `z` at sampled epochs — does the latent *topology* stop rearranging | `z.N.pkl` |
| 3 | **Latent shifts** — median per-particle inter-epoch displacement (magnitude, dot product, cosine distance). The most useful numeric signal: magnitude decaying to ~0 means the encoder has settled | `z.N.pkl` |
| 4 | **Map–map CC + FSC across epochs** — convergence measured on the *structures*, not the loss | `weights.N.pkl` → volumes |

Metrics 1–3 are pure NumPy/scipy and run fine on CPU. Metric 4 regenerates
`final_maxima × len(epochs)` volumes (60 by default) and is ~0.5 PFLOP of decoder MLP —
**seconds per volume on a GPU, minutes per volume on 2 Colab vCPU.** Hence `mode`:
`fast` does 1–3 only; `full` adds metric 4. `auto` picks `full` when a GPU is visible.

> **On overfitting.** `train_vae` has **no held-out validation set** — every particle is in
> every epoch's loss — so a falling loss curve says nothing about generalisation. The proxies
> you do get: the separately-logged **KLD** term (collapsing → the latent is unused; growing
> while reconstruction falls → the latent is absorbing per-particle noise), and metric 4
> (volumes at a fixed latent point still changing at high frequency while loss falls). For a
> real validation loss you need `cryodrgn eval_images` on particles excluded from training via
> `--ind` — a separate side experiment, not something to retrofit onto a full-stack run.

> ⚠️ `analyze_convergence` is marked **BETA** in cryoDRGN and has no test coverage. Cell 7.5
> patches three bugs in it before running:
> 1. It asserts `z.0.pkl` exists, but 4.x training writes `z.1.pkl …` (1-based epochs).
> 2. Its masking step looks for `vol_000.mrc` while `eval_vol` writes `vol_001.mrc …`.
> 3. The UMAP montage reads `epochs[i]` just *outside* its own `try: … except IndexError`, so
>    any number of sampled epochs that doesn't exactly fill a `ceil(√N)`-wide subplot grid
>    crashes on the spare axes — N = 3, 5, 7, 8, 10, 11, 13… all die; only 2, 4, 6, 9, 12, 16,
>    20, 25 survive. The crash lands *after* every UMAP is saved, so 7.5 catches it and redraws.

In [ ]:
#@title 7.4 · Loss / KLD curves from run.log (safe to run mid-training) { display-mode: "form" }
#@markdown Reads `run.log` in place — no GPU, no checkpoints, works while 6.1/6.2 is running.
#@markdown Run folder name (or full path) — leave blank to use the run set by cell 6.
model_folder = ""  #@param {type:"string"}

import os, re
import numpy as np
import matplotlib.pyplot as plt

if model_folder.strip():
    name = model_folder.strip()
    outdir = name if os.path.isabs(name) else os.path.join(os.environ["CRYODRGN_DRIVE_DIR"], name)
else:
    outdir = os.environ.get("CRYODRGN_OUTDIR", "")
if not outdir:
    raise RuntimeError("No model folder — run cell 6.1/6.2 first, or type the run name above.")
logfile = os.path.join(outdir, "run.log")
if not os.path.exists(logfile):
    raise FileNotFoundError(f"{logfile} not found.")

# "# =====> Epoch: 7 Average gen loss = 0.0143, KLD = 12.481, total loss = 0.0159; Finished in ..."
pat = re.compile(
    r"Epoch:\s*(\d+)\s+Average gen loss\s*=\s*([\d.eE+-]+),\s*"
    r"KLD\s*=\s*([\d.eE+-]+),\s*total loss\s*=\s*([\d.eE+-]+)"
)
# dict keyed on epoch: a resumed run re-appends to run.log, and overlapping epochs would
# otherwise be double-counted — last write for an epoch wins.
rows = {}
with open(logfile) as f:
    for line in f:
        m = pat.search(line)
        if m:
            rows[int(m.group(1))] = tuple(float(m.group(i)) for i in (2, 3, 4))

if not rows:
    raise RuntimeError(
        f"No epoch-summary lines in {logfile} yet — the first epoch has not finished. "
        "Per-batch '# [Train Epoch: ...]' lines are logged sooner but are not parsed here."
    )

ep = np.array(sorted(rows))
gen, kld, tot = (np.array([rows[e][i] for e in ep]) for i in range(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, y, name in zip(axes, (tot, gen, kld), ("total loss", "reconstruction (gen) loss", "KLD")):
    ax.plot(ep, y, marker="o", ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"{len(ep)} epochs parsed from {logfile}  (epochs {ep.min()}–{ep.max()})")
if len(ep) >= 6:
    d = (tot[-1] - tot[-6]) / abs(tot[-6]) * 100
    print(f"total loss over the last 5 epochs: {d:+.2f}%  "
          f"({'still improving' if d < -0.5 else 'flattening out'})")
print("KLD trend:", "rising" if kld[-1] > kld[0] else "falling",
      f"({kld[0]:.3f} → {kld[-1]:.3f})")
#@markdown ---
#@markdown Reminder: this is **training** loss on every particle. It cannot show overfitting —
#@markdown see the notes above 7.4 and use `cryodrgn eval_images` on a held-out `--ind` subset
#@markdown if you need a genuine validation number.

In [ ]:
#@title 7.5 · Run analyze_convergence (patches 3 bugs in it) { display-mode: "form" }
#@markdown ### Cells to run first
#@markdown To diagnose a **finished** run — including in a brand-new session — you need only:
#@markdown
#@markdown > **1.1 → 1.2 → 1.3** (install cryoDRGN) → **2.1 → 2.2** (mount Drive, set the same
#@markdown > `drive_project_dir`) → **this cell**, with `model_folder` set to the run's folder
#@markdown > name (e.g. `00_cryodrgn128`). Then **7.6** to view the plots.
#@markdown
#@markdown **Skip Steps 3, 4, 5, 6 and 7.1 entirely.** Nothing here loads the particle stack or
#@markdown the poses, and even `full` mode decodes its volumes from `weights.N.pkl` +
#@markdown `config.yaml`. It does read `ctf.pkl` for the pixel size, but directly from the path
#@markdown recorded in `config.yaml` — so 4.3 need not be re-run — and falls back to 1.0 Å/px
#@markdown with a printed warning if that file has moved.
#@markdown ###### &nbsp;
#@markdown `fast` = metrics 1–3 (CPU-friendly, ~20–40 min). `full` = adds map–map CC/FSC,
#@markdown which regenerates volumes — fine on a GPU, hours on CPU. `auto` picks by GPU presence.
mode = "auto"  #@param ["auto", "fast", "full"]
#@markdown Latest epoch to analyze — **-1** auto-picks the newest `z.N.pkl`.
epoch = -1  #@param {type:"integer"}
#@markdown Epochs between the expensive checks (UMAP / volumes).
epoch_interval = 5  #@param {type:"integer"}
#@markdown Particles subsampled for UMAP (the full stack is far slower for no extra insight).
umap_subset = 50000  #@param {type:"integer"}
#@markdown `full` only: volumes per epoch, and box size for them (`0` = no downsampling).
final_maxima = 10  #@param {type:"integer"}
vol_downsample = 0  #@param {type:"integer"}
#@markdown `full` only: pixel size (Å/px) stamped into the generated volumes. **`0` resolves it
#@markdown from `ctf.pkl`** the way `cryodrgn analyze` does — `analyze_convergence` itself just
#@markdown hardcodes a default of 1.0 and never consults the CTF.
apix = 0  #@param {type:"number"}
#@markdown Reuse UMAPs cached by a previous run of this cell (skips the slowest CPU step).
reuse_umaps = False  #@param {type:"boolean"}
#@markdown Run folder name under your Drive project — leave blank to use the run set by cell 6.
#@markdown Fill it in to diagnose a finished run in a **fresh session**: only Steps 1 and 2 are
#@markdown needed first, since nothing here reads the particle stack, poses or CTF.
model_folder = ""  #@param {type:"string"}

import os, re, glob, shutil, logging, argparse
import numpy as np
import matplotlib.pyplot as plt
import torch

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
from cryodrgn.commands_utils import analyze_convergence as ac
import cryodrgn.analysis as _analysis
import cryodrgn.config

if model_folder.strip():
    name = model_folder.strip()
    outdir = name if os.path.isabs(name) else os.path.join(os.environ["CRYODRGN_DRIVE_DIR"], name)
    if not os.path.exists(os.path.join(outdir, "config.yaml")):
        raise FileNotFoundError(f"No run at {outdir} — check model_folder (it must match 6.1).")
    os.environ["CRYODRGN_OUTDIR"] = outdir      # so 7.6 and Steps 8-9 target this run too
else:
    outdir = os.environ.get("CRYODRGN_OUTDIR", "")
    if not outdir:
        raise RuntimeError(
            "CRYODRGN_OUTDIR unset — run cell 6.1/6.2, or type the run folder name above."
        )

# ---- resolve the latest epoch --------------------------------------------------------
if int(epoch) < 0:
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "z.*.pkl"))
             for m in [re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))] if m]
    found = [e for e in found if e > 0]          # ignore any z.0.pkl we created below
    if not found:
        raise FileNotFoundError(f"No z.N.pkl checkpoints in {outdir}.")
    epoch = max(found)
E = int(epoch)
if E < 4:
    raise ValueError(f"Need at least 4 epochs for convergence heuristics (found {E}).")

epochs = np.arange(4, E + 1, int(epoch_interval))
if epochs[-1] != E:
    epochs = np.append(epochs, E)
# The library's plotting builds subplot grids from these counts and indexes them as 2-D
# arrays; degenerate grids raise a bare AttributeError/IndexError deep inside matplotlib.
if len(epochs) < 2:
    raise ValueError(
        f"Only one sampled epoch ({list(epochs)}) — needs at least 2. Train past epoch 5."
    )
if len(epochs) < 3:
    print(f"  ⚠️  only {len(epochs)} sampled epochs — reduce epoch_interval for a fuller picture.")
if int(final_maxima) < 3:
    raise ValueError(f"final_maxima must be >= 3 (got {final_maxima}).")
print(f"Analyzing '{os.path.basename(outdir)}' up to epoch {E}; sampled epochs: {list(epochs)}")

# ---- patch 1: analyze_convergence asserts z.0.pkl, but training writes z.1.pkl … ----
# encoder_latent_shifts also hard-loads z.0/1/2.pkl. Seed z.0.pkl from z.1.pkl.
z0, z1 = (os.path.join(outdir, f"z.{i}.pkl") for i in (0, 1))
if not os.path.exists(z0):
    if not os.path.exists(z1):
        raise FileNotFoundError(f"{z1} missing — was train_vae run with --checkpoint 1?")
    shutil.copyfile(z1, z0)   # copy, not symlink: Drive's FUSE mount has no symlink support
    print("  patched: seeded z.0.pkl from z.1.pkl — the epoch-2 point of the latent-shift plot\n"
          "           is therefore degenerate (read it from epoch 3 on). z.0.pkl is inert for\n"
          "           resuming (--load latest and cell 6.2 both key off weights.N.pkl) and is\n"
          "           safe to delete afterwards.")

# ---- patch 2: masking looks for vol_000.mrc, eval_vol writes vol_001.mrc … -----------
if not getattr(_analysis.gen_volumes, "_zero_indexed", False):
    _gv = _analysis.gen_volumes
    def _gen_volumes_0(*a, **k):
        return _gv(*a, **{**k, "vol_start_index": 0})
    _gen_volumes_0._zero_indexed = True
    _analysis.gen_volumes = _gen_volumes_0
    ac.analysis.gen_volumes = _gen_volumes_0     # ac imported it by module, but be explicit
    print("  patched: gen_volumes forced to vol_start_index=0 so mask_volumes finds its inputs")

# ---- patch 3: encoder_latent_umaps builds a ceil(sqrt(N)) x ceil(N/ceil(sqrt(N))) subplot
# grid, then reads epochs[i] on the line ABOVE its own `try:` (analyze_convergence.py:306-307).
# So its `except IndexError` never fires and any N that doesn't exactly fill the grid dies on
# the spare axes: N=3,5,7,8,10,11,13.. crash; only 2,4,6,9,12,16,20,25 survive. (The same loop
# in follow_candidate_particles and calculate_FSCs indexes inside the try, so those are fine.)
# Every umap.N.pkl and ind_subset.pkl is written before the crash, so recover and redraw.
def _draw_umap_montage(cdir_, epochs_):
    n = len(epochs_)
    n_cols = int(np.ceil(n ** 0.5))
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2 * n_rows),
                             sharex="all", sharey="all", squeeze=False)
    toplot = None
    for k, ax in enumerate(axes.flat):
        fl = (os.path.join(cdir_, "umaps", f"umap.{epochs_[k]}.pkl")
              if k < n else None)
        if fl is None or not os.path.exists(fl):
            ax.set_visible(False)            # blank the spare axes instead of crashing
            continue
        emb = ac.utils.load_pkl(fl)
        toplot = ax.hexbin(emb[:, 0], emb[:, 1], bins="log", mincnt=1)
        ax.set_title(f"epoch {epochs_[k]}")
    for a in axes[:, 0]:
        a.set_ylabel("UMAP2")
    for a in axes[-1, :]:
        a.set_xlabel("UMAP1")
    fig.subplots_adjust(right=0.96)
    if toplot is not None:
        cbar = fig.colorbar(toplot, cax=fig.add_axes([0.98, 0.15, 0.02, 0.7]))
        cbar.ax.set_ylabel("particle density", rotation=90)
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    out_fl = os.path.join(cdir_, "plots", "01_encoder_umaps.png")
    plt.savefig(out_fl, dpi=300, format="png", transparent=True, bbox_inches="tight")
    plt.close("all")
    print(f"  redrew {out_fl}")

if not getattr(ac.encoder_latent_umaps, "_grid_safe", False):
    _elu = ac.encoder_latent_umaps
    def _encoder_latent_umaps_safe(*a, **k):
        try:
            _elu(*a, **k)
        except IndexError:
            plt.close("all")
            print("  recovered: encoder_latent_umaps' ragged-grid IndexError — every UMAP was\n"
                  "             already computed and saved, redrawing the montage")
            _draw_umap_montage(a[1], a[2])    # (workdir, outdir, epochs, ...) — all positional
    _encoder_latent_umaps_safe._grid_safe = True
    ac.encoder_latent_umaps = _encoder_latent_umaps_safe
    print("  patched: encoder_latent_umaps guarded against its ragged-subplot-grid IndexError")

# ---- decide fast vs full ------------------------------------------------------------
has_gpu = torch.cuda.is_available()
run_mode = mode if mode != "auto" else ("full" if has_gpu else "fast")
print(f"  GPU visible: {has_gpu} → mode = {run_mode}")
if run_mode == "full" and not has_gpu:
    n_vols = int(final_maxima) * len(epochs)
    print(f"  ⚠️  'full' on CPU will generate {n_vols} volumes — expect several HOURS. "
          f"Set vol_downsample=64 to cut that ~8x, or use mode='fast'.")

cdir = os.path.join(outdir, f"convergence.{E}")
for sub in ("plots", "umaps", "repr_particles"):
    os.makedirs(os.path.join(cdir, sub), exist_ok=True)

# ---- can we honour reuse_umaps? -----------------------------------------------------
need = [os.path.join(cdir, "umaps", f"umap.{e}.pkl") for e in epochs]
need.append(os.path.join(cdir, "ind_subset.pkl"))   # follow_candidate_particles needs this
skip_umap = bool(reuse_umaps) and all(os.path.exists(p) for p in need)
if reuse_umaps and not skip_umap:
    print("  reuse_umaps requested but the cache is incomplete — recomputing UMAPs.")

if run_mode == "full":
    # analyze_convergence hardcodes --Apix default 1.0 and never reads ctf.pkl, unlike
    # cryodrgn analyze (analyze.py:458-491). It only reaches the MRC headers of the generated
    # volumes -- the CC is scale-free and the FSC x-axis is in 1/px -- but a wrong header makes
    # the maps unusable in ChimeraX, so resolve it here exactly the way analyze does.
    use_apix = float(apix)
    if use_apix <= 0:
        _cfg = cryodrgn.config.load(os.path.join(outdir, "config.yaml"))
        _ctf = _cfg["dataset_args"].get("ctf")
        if _ctf and os.path.exists(_ctf):
            _cp = ac.utils.load_pkl(_ctf)
            _apixs = set(_cp[:, 1])
            if len(_apixs) > 1:            # multiple optics groups — analyze gives up too
                use_apix = 1.0
                print("  A/px: no unique value in ctf.pkl (multiple optics groups) → 1.0")
            else:
                _orig_apix, _orig_size = tuple(_apixs)[0], tuple(set(_cp[:, 0]))[0]
                _cur = _cfg["lattice_args"]["D"] - 1
                use_apix = round(_orig_apix * _orig_size / _cur, 6)
                print(f"  A/px: {use_apix} from ctf.pkl "
                      f"({_orig_apix} Å/px at box {int(_orig_size)} → box {_cur})")
        else:
            use_apix = 1.0
            print("  A/px: no ctf.pkl recorded in config.yaml → 1.0")

    p = argparse.ArgumentParser()
    ac.add_args(p)
    argv = [outdir, str(E),
            "--outdir", cdir,
            "--epoch-interval", str(int(epoch_interval)),
            "--subset", str(int(umap_subset)),
            "--random-seed", "0",          # else the UMAP subset differs run to run
            "--Apix", str(use_apix),
            "--final-maxima", str(int(final_maxima))]
    if int(vol_downsample) > 0:
        argv += ["-d", str(int(vol_downsample))]
    if skip_umap:
        argv += ["--skip-umap"]
    print("$ cryodrgn_utils analyze_convergence " + " ".join(argv), "\n" + "=" * 70)
    plt.close("all")   # the library reuses the active figure in places; don't overlay a re-run
    ac.main(p.parse_args(argv))
    plt.close("all")
else:
    # Metrics 1-3 only. Calling the functions directly rather than main() with --skip-volgen:
    # masking sits inside the same `else` block as volume generation, so --skip-volgen leaves
    # calculate_CCs with no .masked.mrc to read and it dies after writing plots 00-04.
    print("=" * 70)
    logfile = os.path.join(outdir, "run.log")
    n_total, n_dim = ac.utils.load_pkl(os.path.join(outdir, f"z.{E}.pkl")).shape

    print("Convergence 1: total loss curve ...")
    plt.close("all")     # plot_loss draws onto the active figure; don't overlay a re-run
    plt.figure(figsize=(4, 3))
    ac.plot_loss(logfile, cdir, E)
    plt.close("all")

    if skip_umap:
        print("Convergence 2: reusing cached UMAPs ...")
    else:
        print(f"Convergence 2: UMAP embeddings for epochs {list(epochs)} "
              f"({min(n_total, int(umap_subset))} particles each) ...")
        use_gpu_umap = hasattr(ac, "cuUMAP")
        print("  backend:", "cuML (GPU)" if use_gpu_umap else
              "umap-learn (CPU, single-threaded — random_state is fixed at 42)")
        # random_seed=0 (not None) so the particle subset is reproducible and a later
        # mode='full' run reuses the same one; n_epochs_umap only applies to the cuML path.
        ac.encoder_latent_umaps(outdir, cdir, epochs, n_total, int(umap_subset),
                                0, use_gpu_umap, 42, 25000)

    print(f"Convergence 3: latent encoding shifts over epochs 2-{E} ...")
    plt.close("all")
    with np.errstate(invalid="ignore", divide="ignore"):   # the seeded z.0 makes point 1 NaN
        ac.encoder_latent_shifts(outdir, cdir, E)
    plt.close("all")

    print("Skipping Convergence 4 (volume CC + FSC) — set mode='full' on a GPU runtime.")

print("=" * 70 + f"\n✅ Convergence results → {cdir}\n   View them with cell 7.6.")

In [ ]:
#@title 7.6 · View convergence plots inline { display-mode: "form" }
#@markdown Leave `epoch` at **-1** to show the newest `convergence.N/` folder.
epoch = -1  #@param {type:"integer"}
#@markdown Run folder name — leave blank to use the run set by cell 6 or 7.5.
model_folder = ""  #@param {type:"string"}

import os, re, glob
from IPython.display import Image, display, Markdown

if model_folder.strip():
    name = model_folder.strip()
    outdir = name if os.path.isabs(name) else os.path.join(os.environ["CRYODRGN_DRIVE_DIR"], name)
else:
    outdir = os.environ.get("CRYODRGN_OUTDIR", "")
    if not outdir:
        raise RuntimeError("CRYODRGN_OUTDIR unset — run 7.5, or type the run folder name above.")
if int(epoch) < 0:
    dirs = [(int(m.group(1)), p) for p in glob.glob(os.path.join(outdir, "convergence.*"))
            for m in [re.search(r"convergence\.(\d+)$", p)] if m]
    if not dirs:
        raise FileNotFoundError(f"No convergence.N/ folder in {outdir} — run 7.5 first.")
    epoch, cdir = max(dirs)
else:
    cdir = os.path.join(outdir, f"convergence.{int(epoch)}")

captions = [
    ("00_total_loss.png", "**1 · Total loss** — training loss only; cannot show overfitting"),
    ("01_encoder_umaps.png", "**2 · UMAP of z per epoch** — has the latent topology stopped rearranging?"),
    ("02_encoder_latent_vector_shifts.png",
     "**3 · Latent shifts** — median per-particle displacement between epochs. "
     "*Magnitude* decaying toward 0 is the clearest convergence signal. Ignore the first point "
     "(artifact of the seeded `z.0.pkl`)"),
    ("03_decoder_UMAP-sketching.png", "Latent-space sketching: local maxima chosen for volume generation"),
    ("04_decoder_maxima-sketch-consistency.png", "Sketched classes tracked backwards through training"),
    ("05_decoder_CC.png", "**4a · Masked map–map CC** between consecutive sampled epochs, per class"),
    ("06_decoder_FSC.png",
     "**4b · Masked map–map FSC** between consecutive sampled epochs. The x-axis is spatial "
     "frequency in **1/pixel** (`calc_fsc` returns `bins / D`), *not* 1/Å — convert with "
     "resolution (Å) = A/px ÷ frequency"),
    ("07_decoder_FSC-nyquist.png", "**4b · FSC at Nyquist** — flattening means the maps have stopped changing"),
]

shown = 0
for fname, caption in captions:
    hits = glob.glob(os.path.join(cdir, "plots", fname))
    if hits:
        display(Markdown(caption))
        display(Image(hits[0], width=620))
        shown += 1

print(f"{shown} plot(s) from {cdir}/plots")
if shown and shown <= 3:
    print("Only metrics 1-3 present — that is the 'fast' mode output. "
          "Re-run 7.5 with mode='full' on a GPU runtime for the map-map CC/FSC panels.")

## 8 · (Optional) Filter particles & retrain

`analyze` usually reveals junk particles (bad clusters, latent-space outliers) that are worth
removing before retraining on the clean subset. cryoDRGN's *interactive* lasso filter
(`cryodrgn filter` and the filtering notebook) forces matplotlib's `TkAgg` desktop backend,
which Colab can't provide — so there are **two routes**, and every selection produces an
`indices.pkl` that feeds the same `--ind` retrain (8.3):

- **(a) Select in Colab → retrain.** Either **deterministic rules** (8.1 — cluster IDs, ‖z‖
  outliers, or a PC/UMAP range; reproducible and always works) or an **experimental interactive
  lasso** (8.2 — draw right on the plot). Then retrain on the result with 8.3.
- **(b) Interactive filtering on your local machine (8.4).** Package everything `cryodrgn filter`
  (or the local filtering notebook) needs into one zip on Drive, lasso locally, then bring
  `indices.pkl` back and retrain with 8.3.

> These cells assume the **first** filtering pass (model trained on the full stack). Iterative
> multi-round filtering needs index composition — see the
> [user guide](https://ez-lab.gitbook.io/cryodrgn/).

In [ ]:
#@title 8.1 · Deterministic selection → indices.pkl (in Colab) { display-mode: "form" }
#@markdown Pick particles by a **rule** and write `indices.pkl` into your model folder.
method = "Keep k-means clusters"  #@param ["Keep k-means clusters", "Remove z-norm outliers", "Keep by axis range"]
#@markdown • *Keep k-means clusters* — comma-separated cluster IDs, **0-based**, matching the
#@markdown values in `kmeans*/labels.pkl` and the colour bar of the k-means plot in 7.2.
#@markdown Note the volume files are numbered from 1, so cluster `i` is `vol_{i+1:03d}.mrc`
#@markdown (`cryodrgn analyze` defaults to `--vol-start-index 1`). The cell lists the mapping.
clusters_to_keep = "0,1,2"  #@param {type:"string"}
#@markdown • *Remove z-norm outliers* — drop particles whose ‖z‖ exceeds mean + (this)·std.
outlier_std = 2.0  #@param {type:"number"}
#@markdown • *Keep by axis range* — keep particles whose coordinate lies within [min, max].
axis = "PC1"  #@param ["PC1", "PC2", "UMAP1", "UMAP2"]
axis_min = -3.0  #@param {type:"number"}
axis_max = 3.0  #@param {type:"number"}

import os, re, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn import analysis, utils

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:   # 7.1 skipped, or a fresh session — take the newest analyze.N/
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
z = utils.load_pkl(os.path.join(outdir, f"z.{epoch}.pkl"))
z = z.reshape(z.shape[0], -1)
N = z.shape[0]
adir = os.path.join(outdir, f"analyze.{epoch}")

# PCA is always available; UMAP only exists for zdim > 2
pc = analysis.run_pca(z)[0] if z.shape[1] > 1 else z
umap_fl = os.path.join(adir, "umap.pkl")
umap = utils.load_pkl(umap_fl) if os.path.exists(umap_fl) else None

if method == "Keep k-means clusters":
    lbls = glob.glob(os.path.join(adir, "kmeans*", "labels.pkl"))
    if not lbls:
        raise FileNotFoundError("No k-means labels found — run 7.1 (analyze) first.")
    labels = np.asarray(utils.load_pkl(lbls[0])).ravel()
    keep = [int(x) for x in clusters_to_keep.split(",") if x.strip() != ""]
    if not keep:
        raise ValueError("clusters_to_keep is empty.")
    present = np.unique(labels)
    # analyze writes labels.pkl 0-based (analyze.py:162) but numbers its volumes from 1
    # (--vol-start-index default 1), and cryodrgn_utils select_clusters takes 1-based IDs.
    # Three conventions for one thing, so say plainly which one this field uses.
    unknown = sorted(set(keep) - set(present.tolist()))
    if unknown:
        raise ValueError(
            f"Cluster ID(s) {unknown} are not in {os.path.basename(os.path.dirname(lbls[0]))}"
            f"/labels.pkl, which holds {present.min()}..{present.max()}.\n"
            f"  These IDs are 0-based. If you read them off the volume filenames, subtract 1:\n"
            f"  vol_{unknown[0]:03d}.mrc is cluster {unknown[0] - 1}. (cryodrgn_utils "
            f"select_clusters uses the 1-based form; this field does not.)")
    mask = np.isin(labels, keep)
    print(f"keeping clusters {sorted(keep)} of {present.min()}..{present.max()}  "
          f"→ volume files " + ", ".join(f"vol_{c + 1:03d}.mrc" for c in sorted(keep)[:6])
          + (" ..." if len(keep) > 6 else ""))
elif method == "Remove z-norm outliers":
    znorm = np.linalg.norm(z, axis=1)
    mask = znorm <= (znorm.mean() + float(outlier_std) * znorm.std())
else:
    arr, col = {"PC1": (pc, 0), "PC2": (pc, 1), "UMAP1": (umap, 0), "UMAP2": (umap, 1)}[axis]
    if arr is None or col >= arr.shape[1]:
        raise ValueError(f"{axis} is unavailable for this model (UMAP needs zdim>2; PC2 needs zdim>=2).")
    coord = arr[:, col]
    mask = (coord >= float(axis_min)) & (coord <= float(axis_max))

ind_keep = np.where(mask)[0]
out_ind = os.path.join(outdir, "indices.pkl")
utils.save_pkl(ind_keep, out_ind)
print(f"Keeping {len(ind_keep)} / {N} particles ({100 * len(ind_keep) / N:.1f}%).")
print(f"✅ Saved → {out_ind}   (retrain with cell 8.3)")

# Show the split wherever a 2-D embedding is available
panels = [(X, name) for X, name in [(pc, "PCA"), (umap, "UMAP")]
          if X is not None and X.shape[1] >= 2]
if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 4.5), squeeze=False)
    for ax, (X, name) in zip(axes[0], panels):
        ax.scatter(X[~mask, 0], X[~mask, 1], s=2, alpha=.2, color="lightgray", rasterized=True, label="removed")
        ax.scatter(X[mask, 0], X[mask, 1], s=2, alpha=.3, color="tab:blue", rasterized=True, label="kept")
        ax.set_title(name); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=4, loc="best")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 8.2 · Interactive lasso selection (in Colab, experimental) → indices.pkl { display-mode: "form" }
#@markdown Draw a lasso directly on the latent scatter; on release the selection saves to
#@markdown `indices.pkl` automatically (watch for the printed confirmation). This uses Colab's
#@markdown JS→Python bridge instead of plotly's `FigureWidget` callback, which does **not** fire
#@markdown in Colab. **If nothing prints when you lasso, your session's bridge isn't wired —
#@markdown just use the deterministic cell 8.1 instead.**
max_display_points = 50000  #@param {type:"integer"}

import os, re, glob
import numpy as np
import plotly.graph_objects as go
from google.colab import output
from IPython.display import HTML, display
from cryodrgn import analysis, utils

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:   # 7.1 skipped, or a fresh session — take the newest analyze.N/
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
adir = os.path.join(outdir, f"analyze.{epoch}")
z = utils.load_pkl(os.path.join(outdir, f"z.{epoch}.pkl"))
z = z.reshape(z.shape[0], -1)
N = z.shape[0]

# 2-D embedding for the scatter: prefer UMAP (zdim > 2), else fall back to PCA
umap_fl = os.path.join(adir, "umap.pkl")
if os.path.exists(umap_fl):
    emb, emb_name = utils.load_pkl(umap_fl), "UMAP"
elif z.shape[1] >= 2:
    emb, emb_name = analysis.run_pca(z)[0][:, :2], "PCA"
else:
    raise ValueError("Need zdim>=2 (or a completed UMAP) for a 2-D lasso — use cell 8.1 instead.")

lbls = glob.glob(os.path.join(adir, "kmeans*", "labels.pkl"))
labels = utils.load_pkl(lbls[0]) if lbls else np.zeros(N, dtype=int)

# subsample for a responsive browser plot; keep the map from display index -> original index
rng = np.random.default_rng(0)
sub = (np.sort(rng.choice(N, int(max_display_points), replace=False))
       if N > int(max_display_points) else np.arange(N))

out_ind = os.path.join(outdir, "indices.pkl")
def _save_selection(point_inds):
    orig = np.sort(sub[np.asarray(point_inds, dtype=int)])
    utils.save_pkl(orig.astype(int), out_ind)
    print(f"✅ Saved {len(orig)} / {N} particles → {out_ind}   (retrain with cell 8.3)")
output.register_callback("cryodrgn.save_selection", _save_selection)

fig = go.Figure(go.Scattergl(
    x=emb[sub, 0], y=emb[sub, 1], mode="markers",
    marker=dict(color=labels[sub], size=3, colorscale="Turbo",
                showscale=True, colorbar=dict(title="kmeans")),
))
fig.update_layout(dragmode="lasso", width=680, height=560,
                  title=f"{emb_name} — lasso to select ({len(sub)} of {N} shown)",
                  xaxis_visible=False, yaxis_visible=False, margin=dict(l=0, r=0, t=40, b=0))

div = "cryodrgn_lasso"
html = fig.to_html(include_plotlyjs="cdn", full_html=False, div_id=div)
bridge_lines = [
    "<script>",
    "(function attach(){",
    f"  var gd = document.getElementById('{div}');",
    "  if (!gd || !gd.on) { return setTimeout(attach, 200); }",
    "  gd.on('plotly_selected', function(e){",
    "    if (!e) return;",
    "    var inds = e.points.map(function(p){ return p.pointIndex; });",
    "    google.colab.kernel.invokeFunction('cryodrgn.save_selection', [inds], {});",
    "  });",
    "})();",
    "</script>",
]
display(HTML(html + "\n".join(bridge_lines)))
print("Draw a lasso on the plot above; release to save. Re-lasso to replace the selection.")

In [ ]:
#@title 8.3 · Retrain on the filtered particles { display-mode: "form" }
#@markdown Trains a fresh model on only the kept particles via `--ind`. Leave `indices_pkl`
#@markdown blank to use the `indices.pkl` from 8.1 / 8.2, or paste a path to one you made locally
#@markdown (e.g. uploaded to Drive from a `cryodrgn filter` session, or written by the
#@markdown export-subset notebook). A `.txt` of one index per line is accepted too.
indices_pkl = ""  #@param {type:"string"}
#@markdown Particle stack — blank uses the one cell 4.1 resolved. Set it to train on a specific
#@markdown file, e.g. a single-wave `particles.128.0.mrcs`.
particles = ""  #@param {type:"string"}
zdim = 8  #@param [1, 2, 4, 8, 10] {type:"raw"}
num_epochs = 25  #@param {type:"integer"}
batch_size = 16  #@param [8, 16, 32] {type:"raw"}
output_name = "01_cryodrgn128_filtered"  #@param {type:"string"}
#@markdown **`inherit`** copies the dataset flags from `source_run`'s `config.yaml`. Getting
#@markdown `--uninvert-data` wrong inverts every image and ruins the run with no error, so
#@markdown inheriting from the run you are filtering is the safe choice.
uninvert_data = "inherit"  #@param ["inherit", "yes", "no"]
#@markdown Run to inherit from — a folder name under your Drive project (e.g. `00_cryodrgn128`),
#@markdown or blank to use whichever run cell 6/7 last selected. Only read when `inherit`.
source_run = ""  #@param {type:"string"}
#@markdown Lazy loading. **`auto`** keeps it off unless the *kept* particles will not fit in RAM
#@markdown as float32 — filtering shrinks the resident array, so a subset often fits when the
#@markdown parent stack did not.
lazy = "auto"  #@param ["auto", "no", "yes"]
#@markdown RAM to leave free for everything that is not the particle array. Only used by `auto`.
ram_headroom_gib = 4  #@param {type:"number"}
#@markdown DataLoader workers. **`-1` = auto** (0 on a GPU runtime — see 6.1), `0` = main
#@markdown process, `N` = that many. Eager workers also cost ~0.6 GiB each.
num_workers = -1  #@param {type:"integer"}

import os, glob, pickle
import numpy as np
from cryodrgn import utils
from cryodrgn.source import ImageSource

ds = particles.strip() or os.environ["CRYODRGN_DOWNSAMPLED"]
pose = os.environ["CRYODRGN_POSE"]
ctf = os.environ["CRYODRGN_CTF"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
if not os.path.exists(ds):
    raise FileNotFoundError(f"Particle stack not found: {ds}")

ind = indices_pkl.strip() or os.path.join(os.environ["CRYODRGN_OUTDIR"], "indices.pkl")
if not os.path.exists(ind):
    raise FileNotFoundError(f"No index file at {ind} — run 8.1/8.2 or set indices_pkl.")

# Validate the selection against the stack it will be applied to. cryoDRGN would otherwise
# fail deep inside the data loader, or worse, silently index a stack it does not belong to.
n_stack = ImageSource.from_file(ds, lazy=True).n
if ind.lower().endswith(".txt"):
    sel = np.loadtxt(ind, dtype=np.int64, ndmin=1)
    ind_pkl = os.path.splitext(ind)[0] + ".pkl"      # --ind is read with utils.load_pkl
    utils.save_pkl(np.sort(np.asarray(sel).ravel().astype(np.int64)), ind_pkl)
    print(f"converted {os.path.basename(ind)} → {os.path.basename(ind_pkl)} (--ind needs a .pkl)")
    ind = ind_pkl
sel = np.asarray(utils.load_pkl(ind)).ravel()
if sel.dtype == bool:
    sel = np.where(sel)[0]
sel = sel.astype(np.int64)
if sel.size == 0:
    raise ValueError(f"{ind} is empty.")
if sel.min() < 0 or sel.max() >= n_stack:
    raise ValueError(
        f"Indices run {sel.min():,}..{sel.max():,} but {os.path.basename(ds)} holds only "
        f"{n_stack:,} particles. Are these indices for a different stack? A single wave file "
        f"holds one wave, not the whole stack — point `particles` at the .txt index instead."
    )
if np.unique(sel).size != sel.size:
    raise ValueError(f"{ind} contains duplicate indices.")
print(f"stack     : {ds}  ({n_stack:,} particles)")
print(f"selection : {sel.size:,} kept, {n_stack - sel.size:,} removed "
      f"({sel.size / n_stack * 100:.2f}% retained)")

# These live in dataset_args rather than in the model, so a mismatch produces no error --
# it just trains on differently-preprocessed images. Carry them from the source run.
extra = ""
if uninvert_data == "inherit":
    import yaml
    src = source_run.strip() or os.environ.get("CRYODRGN_OUTDIR", "")
    if not src:
        raise RuntimeError(
            "uninvert_data='inherit' needs a run to inherit from, but source_run is blank and\n"
            "  CRYODRGN_OUTDIR is unset (nothing in cells 1-4 sets it — only 6.1/6.2/7.5 do).\n"
            "  Either put the run's folder name in source_run above (e.g. '00_cryodrgn128'),\n"
            "  or set uninvert_data to 'yes'/'no' explicitly. Refusing to guess: guessing wrong\n"
            "  inverts every image and ruins the run silently."
        )
    src_dir = src if os.path.isabs(src) else os.path.join(DRIVE_DIR, src)
    src_cfg = os.path.join(src_dir, "config.yaml")
    if not os.path.exists(src_cfg):
        raise FileNotFoundError(
            f"No config.yaml at {src_cfg} — cannot inherit. Check source_run, or set "
            f"uninvert_data to 'yes'/'no' explicitly."
        )
    da = yaml.safe_load(open(src_cfg))["dataset_args"]
    use_uninvert = da.get("invert_data") is False
    print(f"uninvert  : {'ON' if use_uninvert else 'off'} (inherited from {src_cfg})")
    # the rest of the preprocessing, so the filtered model stays comparable to its parent
    if da.get("window") is False:
        extra += " --no-window"
        print("            + --no-window (inherited)")
    if da.get("window_r") not in (None, 0.85):
        extra += f' --window-r {da["window_r"]}'
        print(f"            + --window-r {da['window_r']} (inherited)")
    if da.get("datadir"):
        extra += f' --datadir "{da["datadir"]}"'
        print(f"            + --datadir (inherited)")
else:
    use_uninvert = uninvert_data == "yes"
    print(f"uninvert  : {'ON' if use_uninvert else 'off'} (set explicitly)")

outdir = os.path.join(DRIVE_DIR, output_name)
if glob.glob(os.path.join(outdir, "weights.*.pkl")):
    raise FileExistsError(
        f"{outdir} already holds checkpoints.\n"
        f"  To CONTINUE that run (e.g. it was cut short by a disconnect), use cell 6.2 with\n"
        f"  output_name='{output_name}'. It rebuilds --ind and every dataset flag from that\n"
        f"  run's config.yaml, so the filtered subset is preserved — do not re-run 8.3, and do\n"
        f"  not use 6.1, which has no --ind and would train on the whole stack.\n"
        f"  To start a DIFFERENT filtered model, change output_name above."
    )

_n_load = int(sel.size)          # --ind means only the kept particles become resident
_ind_arr = sel                   # ...and a non-contiguous one halves the load peak on .mrcs
# --- lazy loading: on only when the eager array will not fit -------------------------
# ImageDataset materializes the SELECTED particles as float32 (source.py:116-126), so the
# steady-state cost is n * D^2 * 4 bytes. The PEAK during the load is what actually decides
# whether the runtime survives, and it is 2x that in every case but one -- measured, not
# assumed:
#   .mrcs, no/contiguous --ind  2.00x   contiguous branch reads the whole stack with one
#                                       np.fromfile and copies it into the preallocated
#                                       array, so both exist at once (source.py:419-422)
#   .txt,  no/contiguous --ind  2.01x   one future per member file, all held in `to_do`
#                                       until the read finishes (source.py:511-528)
#   .txt,  filtering --ind      2.01x   same -- each future still materializes its file's
#                                       whole share, so their sum is the full stack
#   .mrcs, filtering --ind      1.00x   non-contiguous indices take the per-image loop
#                                       (source.py:425-432): one image in flight at a time
# So it is NOT "single file is cheaper" in general -- a plain full load costs 2x either way.
# The saving is specific to a single .mrcs read through a non-contiguous --ind, which is
# exactly the filtered-retraining case. Lazy avoids the peak entirely, but then re-reads
# every image from disk every epoch, which on a Drive-backed stack is far slower. Hence:
# off unless it must be on.
import psutil
import numpy as np
from cryodrgn.source import ImageSource
_D = ImageSource.from_file(ds, lazy=True).D
_single = os.path.splitext(ds)[1].lower() in (".mrc", ".mrcs")
_contig = True
try:
    _ia = np.asarray(_ind_arr).ravel() if _ind_arr is not None else None
    if _ia is not None and _ia.size and not np.issubdtype(_ia.dtype, np.bool_):
        _contig = bool(np.all(_ia == _ia[0] + np.arange(_ia.size)))
except Exception:
    _contig = True                      # unsure -> assume the expensive path
_mult = 1 if (_single and not _contig) else 2
_need = _n_load * _D * _D * 4 * _mult
_avail = psutil.virtual_memory().available
_budget = _avail - float(ram_headroom_gib) * 2**30
use_lazy = (_need > _budget) if lazy == "auto" else (lazy == "yes")

_why = ("  (x2: multi-file loader holds every per-file array)" if _mult == 2 and not _single
        else "  (x2: contiguous read builds the whole stack before copying it)"
        if _mult == 2 else "  (x1: single file + non-contiguous --ind reads one image at a time)")
print(f"eager RAM : {_need / 2**30:.1f} GiB for {_n_load:,} x {_D}^2 float32" + _why)
print(f"            {_avail / 2**30:.1f} GiB free now, {_budget / 2**30:.1f} GiB budget "
      f"after {ram_headroom_gib} GiB headroom")
print(f"lazy      : {'ON' if use_lazy else 'off'}  "
      + (("(auto: the stack does not fit)" if use_lazy else "(auto: the stack fits)")
         if lazy == "auto" else "(set explicitly)"))
if use_lazy:
    print("            lazy re-reads every image each epoch — expect slower epochs. A")
    print("            High-RAM runtime, or a smaller box, would let you keep it off.")
    # The one case with a cheap fix: a chunked .txt read through a filtering --ind pays 2x
    # only because of the per-file futures. The same particles in one .mrcs take the
    # per-image path instead and cost 1x — which here would be enough to stay eager.
    if not _single and not _contig and (_need // 2) <= _budget:
        print(f"            💡 this stack is {os.path.splitext(ds)[1]} (multi-file). Consolidating it into a")
        print(f"            single .mrcs — cell 6.1 of the preprocess notebook — drops the load")
        print(f"            peak to {_need / 2 / 2**30:.1f} GiB, which fits, so this run could stay eager.")
        print(f"            Particle order and every index are preserved, so --ind/poses/ctf")
        print(f"            carry over unchanged.")
elif lazy == "auto" and _need > _budget * 0.8:
    print(f"            ⚠️  only {(_budget - _need) / 2**30:.1f} GiB spare. If the runtime dies")
    print("            with no traceback, that was the OOM killer — set lazy='yes'.")

# With --lazy every image is re-read and re-transformed every epoch, and train_vae gives that
# job just TWO worker processes (train_vae.py:874-880). On a multi-core runtime that, not the
# GPU, is what sets the step time. Each worker also fans out over max_threads for multi-file
# sources, so leave headroom rather than claiming every core.
_ncpu = os.cpu_count() or 2
try:
    import torch as _t
    _cuda = _t.cuda.is_available()
except Exception:
    _cuda = False

# On a GPU runtime train_vae builds ImageDataset with device='cuda' (train_vae.py:693), and
# __getitem__ does .to(self.device) BEFORE _process (dataset.py:128) -- so the transform runs
# on the GPU and every worker returns a CUDA tensor. Handing those across a process boundary
# goes through CUDA IPC, which PyTorch's own docs advise against; it costs an IPC handle per
# batch and a CUDA context per worker, and it is what emits
#   "Producer process has been terminated before all shared CUDA tensors released".
# Measured on an L4 at box 192: 2 workers and 8 workers both give ~17 s/1000 particles, so
# the extra processes buy nothing while adding that per-batch cost. Default to 0 on a GPU.
# -1 = auto; 0 means literally zero, matching DataLoader's own num_workers semantics.
# (An earlier version overloaded 0 as "auto", which read as "no workers" and silently
# gave 8 of them -- exactly the case this whole note is about.)
_nw = int(num_workers)
if _nw >= 0:
    n_workers = _nw
    print(f"workers   : {n_workers} (set explicitly) of {_ncpu} CPU(s)"
          + ("  — main process loads, no IPC" if n_workers == 0 else ""))
    if n_workers and _cuda:
        print("            ⚠️  the dataset is CUDA-resident, so workers return CUDA tensors")
        print("            over IPC — a per-batch cost that does not shrink as you add more.")
        print("            If steps are slow, set num_workers = 0.")
elif _cuda:
    n_workers = 0
    print(f"workers   : 0 (auto — the dataset lives on the GPU, so extra processes would")
    print(f"            only add CUDA IPC per batch; {_ncpu} CPU(s) available if you want to")
    print(f"            override)")
elif use_lazy:
    n_workers = min(8, max(2, _ncpu - 2))
    print(f"workers   : {n_workers} of {_ncpu} CPU(s)  (auto — CPU-only run, so the dataset "
          f"is CPU-resident and workers genuinely parallelise the load)")
else:
    n_workers = 0     # eager on CPU: cryoDRGN loads in the main process

if not use_lazy and n_workers:
    # Workers fork, so the resident array is shared copy-on-write and is NOT duplicated --
    # images() hands out a fancy-indexed copy (source.py:256), so _process's in-place ops
    # never write back to it. What each worker does cost is its own interpreter and
    # buffers, and in eager mode that comes out of the budget the array has already nearly
    # filled. They are still worth it: _process is the only CPU work left once the stack is
    # resident, and cryoDRGN leaves it single-threaded here.
    _wcost = n_workers * 0.6 * 2**30
    _left = _budget - _need - _wcost
    print(f"            + ~{_wcost / 2**30:.1f} GiB for {n_workers} eager worker(s) "
          f"-> {_left / 2**30:.1f} GiB would remain")
    if _left < 0:
        raise MemoryError(
            f"{n_workers} eager workers need ~{_wcost / 2**30:.1f} GiB on top of the "
            f"{_need / 2**30:.1f} GiB array, which overruns the "
            f"{_budget / 2**30:.1f} GiB budget.\n  Lower num_workers, or set it to 0 to let "
            f"the main process load as cryoDRGN does by default.")

cmd = (f'cryodrgn train_vae "{ds}" --poses "{pose}" --ctf "{ctf}" '
       f'--ind "{ind}" --zdim {zdim} -n {int(num_epochs)} -b {int(batch_size)} -o "{outdir}"')
if use_uninvert:
    cmd += " --uninvert-data"
if use_lazy:
    cmd += " --lazy"
cmd += f" --num-workers {n_workers}"
cmd += extra
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")

os.environ["CRYODRGN_OUTDIR"] = outdir  # Step 7 (analyze) now targets the filtered model
print("=" * 70)
print(f"✅ Filtered model → {outdir}")
print("   Re-run Step 7 (cell 7.1) — it now points at this filtered model.")

In [ ]:
#@title 8.4 · Package for interactive filtering on your local machine { display-mode: "form" }
#@markdown Bundles everything `cryodrgn filter` (or the local filtering notebook) needs into one
#@markdown zip on Drive, plus a `fix_paths.py` that repairs the paths for your machine. Download
#@markdown it, filter with the lasso locally, then bring `indices.pkl` back and retrain with 8.2.
include_particle_stack = False  #@param {type:"boolean"}
#@markdown ↑ Needed only if you trained **without CTF**, or you want the notebook's "View
#@markdown particles" montage. The stack can be several GB.

import os, re, glob, shutil, yaml
from cryodrgn import config as cryodrgn_config

outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:   # 7.1 skipped, or a fresh session — take the newest analyze.N/
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

cfg = cryodrgn_config.load(os.path.join(outdir, "config.yaml"))
da = cfg["dataset_args"]

def _locate(p):
    # tolerate Colab reconnects: try the config path, then local scratch / Drive / workdir
    if isinstance(p, str) and os.path.exists(p):
        return p
    base = os.path.basename(p) if isinstance(p, str) else None
    for d in (WORK_DIR, DRIVE_DIR, outdir):
        if base and os.path.exists(os.path.join(d, base)):
            return os.path.join(d, base)
    return None

# Stage a lean copy of the workdir: filter needs z / config / run.log + analyze umap & kmeans
# labels, but NOT the network weights or the generated .mrc volumes.
stage = os.path.join(WORK_DIR, f"filter_bundle_epoch{epoch}")
if os.path.exists(stage):
    shutil.rmtree(stage)
shutil.copytree(outdir, stage,
                ignore=shutil.ignore_patterns("weights*.pkl", "*.mrc", "*.mrcs"))

# Copy pose / ctf / ind (and optionally the stack) next to config.yaml; rewrite config to
# basenames so `cryodrgn filter .` works from inside the folder.
missing = []
for key in ("poses", "ctf", "ind", "particles"):
    if key == "particles" and not include_particle_stack:
        if isinstance(da.get(key), str):
            da[key] = os.path.basename(da[key])  # neutralize dead /content path (unused w/ CTF)
        continue
    src = _locate(da.get(key))
    if src:
        shutil.copy2(src, os.path.join(stage, os.path.basename(src)))
        da[key] = os.path.basename(src)
    elif isinstance(da.get(key), str):
        missing.append(key)

with open(os.path.join(stage, "config.yaml"), "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# fix_paths.py upgrades the basenames to absolute paths, so the CLI and the local filtering
# notebook (whose working directory differs) both resolve regardless of where they launch.
fix_lines = [
    "# fix_paths.py - run once after unzipping (from anywhere):  python fix_paths.py",
    "# Repairs config.yaml so `cryodrgn filter` and the filtering notebook find pose.pkl /",
    "# ctf.pkl (and the particle stack, if bundled) on THIS machine.",
    "import os, yaml",
    "here = os.path.dirname(os.path.abspath(__file__))",
    "cfg_path = os.path.join(here, 'config.yaml')",
    "with open(cfg_path) as f:",
    "    cfg = yaml.safe_load(f)",
    "da = cfg['dataset_args']",
    "for key in ('poses', 'ctf', 'ind', 'particles'):",
    "    v = da.get(key)",
    "    if isinstance(v, str):",
    "        local = os.path.join(here, os.path.basename(v))",
    "        if os.path.exists(local):",
    "            da[key] = local",
    "p = da.get('particles')",
    "if isinstance(p, str) and os.path.exists(os.path.join(here, os.path.basename(p))):",
    "    da['datadir'] = here",
    "with open(cfg_path, 'w') as f:",
    "    yaml.safe_dump(cfg, f, sort_keys=False)",
    "print('Updated config.yaml paths for', here)",
]
with open(os.path.join(stage, "fix_paths.py"), "w") as f:
    f.write("\n".join(fix_lines) + "\n")

zip_base = os.path.join(DRIVE_DIR, f"filter_bundle_epoch{epoch}")
shutil.make_archive(zip_base, "zip", stage)
size_mb = os.path.getsize(zip_base + ".zip") / 1e6
shutil.rmtree(stage)

print(f"✅ Bundle → {zip_base}.zip  ({size_mb:.0f} MB)")
if missing:
    print(f"⚠️  Could not find: {', '.join(missing)} — re-run the matching Step 4 cell to regenerate.")
if da.get("ctf") is None and not include_particle_stack:
    print("⚠️  This model was trained WITHOUT CTF, so filter must open the particle stack —")
    print("    re-run this cell with include_particle_stack = True.")
print("\nOn your local machine (with cryodrgn installed):")
print(f"    unzip filter_bundle_epoch{epoch}.zip -d filter_bundle")
print( "    cd filter_bundle")
print( "    python fix_paths.py        # repair paths for this machine")
print( "    cryodrgn filter .          # interactive lasso  ->  indices.pkl")
print(f"    #  ...or open analyze.{epoch}/cryoDRGN_filtering.ipynb in local Jupyter (set EPOCH, KMEANS)")
print("\nThen upload indices.pkl to your Drive project folder and retrain with cell 8.3.")

## 9 · (Optional) Generate volumes, trajectories & landscape

Decode structures from the trained decoder: a **single volume** at a chosen `z` (9.1), a
**trajectory** — a path through latent space rendered as a volume series / movie (9.2–9.3), or a
full **conformational landscape** analysis (9.4).

In [ ]:
#@title 9.1 · Decode a volume at a specific z { display-mode: "form" }
#@markdown Space-separated latent coordinate, length == `zdim` (e.g. `0.5 -1.2 0 0 0 0 0 0`).
z_value = "0 0 0 0 0 0 0 0"  #@param {type:"string"}
#@markdown Output filename (saved in your Drive project folder).
output_name = "my_volume.mrc"  #@param {type:"string"}
#@markdown Pixel size (Å/px) for the header — **`0` resolves it from `ctf.pkl`**.
apix = 0  #@param {type:"number"}

import os
outdir = os.environ["CRYODRGN_OUTDIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]
weights = os.path.join(outdir, "weights.pkl")
config = os.path.join(outdir, "config.yaml")
out_mrc = os.path.join(DRIVE_DIR, output_name)

# eval_vol declares --Apix with default=1 and, unlike `cryodrgn analyze`, never consults the
# CTF. Resolve it the way analyze does (analyze.py:462-488) so downsampled runs aren't stamped
# with 1.0 A/px — for a 400px stack trained at 128 that is off by 400/128 = 3.125x.
use_apix, apix_src = float(apix), "form field"
if use_apix <= 0:
    import cryodrgn.config
    from cryodrgn import utils as _cdutils
    _cfg = cryodrgn.config.load(config)
    _ctf = _cfg["dataset_args"].get("ctf")
    use_apix, apix_src = 1.0, "fallback — no ctf.pkl in config.yaml"
    if _ctf and os.path.exists(_ctf):
        _cp = _cdutils.load_pkl(_ctf)
        _ap = set(_cp[:, 1])
        if len(_ap) > 1:
            apix_src = "fallback — multiple optics groups"
        else:
            use_apix = round(tuple(_ap)[0] * tuple(set(_cp[:, 0]))[0]
                             / (_cfg["lattice_args"]["D"] - 1), 6)
            apix_src = "from ctf.pkl"
print(f"A/px = {use_apix}  ({apix_src})")

cmd = f'cryodrgn eval_vol "{weights}" --config "{config}" -z {z_value} -o "{out_mrc}"'
cmd += f" --Apix {use_apix}"

print("$", cmd, "\n")
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print(f"\n✅ Volume → {out_mrc}")

In [ ]:
#@title 9.2 · Generate a latent trajectory (z-path) { display-mode: "form" }
#@markdown Build a path through latent space that 9.3 renders into a volume series (a movie).
#@markdown • **graph** — shortest path visiting the anchors along the data manifold (best for movies)
#@markdown • **direct** — straight-line interpolation between anchors • **pc** — along a principal component
method = "graph"  #@param ["graph", "direct", "pc"]
#@markdown **Anchors** (graph/direct) — blank = visit all k-means cluster centers; or a
#@markdown comma-separated list of **particle indices** (e.g. from a `centers_ind.txt`). Unused for `pc`.
anchors = ""  #@param {type:"string"}
#@markdown Points to sample — between each pair of anchors (`direct`) or along the PC (`pc`).
n_points = 10  #@param {type:"integer"}
#@markdown Which principal component, 1-based (`pc` method only).
pc = 1  #@param {type:"integer"}

import os, re, glob
outdir = os.environ["CRYODRGN_OUTDIR"]
epoch = os.environ.get("CRYODRGN_EPOCH") or ""
if not epoch:   # 7.1 skipped, or a fresh session — take the newest analyze.N/
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"No analyze.N/ folder in {outdir} — run 6.1/6.2 or 7.1 first.")
    epoch = str(max(found))
    os.environ["CRYODRGN_EPOCH"] = epoch
    print(f"Auto-selected analyze.{epoch} (CRYODRGN_EPOCH was unset)")
zfile = os.path.join(outdir, f"z.{epoch}.pkl")
adir = os.path.join(outdir, f"analyze.{epoch}")
traj_dir = os.path.join(outdir, f"trajectory.{epoch}")
os.makedirs(traj_dir, exist_ok=True)

if method == "pc":
    cmd = f'cryodrgn pc_traversal "{zfile}" --pc {int(pc)} -n {int(n_points)} -o "{traj_dir}"'
    zpath = os.path.join(traj_dir, f"pc{int(pc)}.txt")
else:
    if anchors.strip():
        anch = anchors.replace(",", " ")
    else:
        ci = glob.glob(os.path.join(adir, "kmeans*", "centers_ind.txt"))
        if not ci:
            raise FileNotFoundError("No k-means centers_ind.txt — run 7.1 (analyze) first, or set anchors.")
        anch = f'"{ci[0]}"'
    zpath = os.path.join(traj_dir, "z-path.txt")
    if method == "graph":
        outind = os.path.join(traj_dir, "z-path-indices.txt")
        cmd = f'cryodrgn graph_traversal "{zfile}" --anchors {anch} -o "{zpath}" --outind "{outind}"'
    else:
        cmd = f'cryodrgn direct_traversal "{zfile}" --anchors {anch} -n {int(n_points)} -o "{zpath}"'

print("$", cmd, "\n")
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
os.environ["CRYODRGN_ZPATH"] = zpath
npts = sum(1 for _ in open(zpath)) if os.path.exists(zpath) else 0
print(f"\n✅ z-path ({npts} points) → {zpath}   (render with 9.3)")

In [ ]:
#@title 9.3 · Render the trajectory to a volume series { display-mode: "form" }
#@markdown Decodes every z on the path into an `.mrc`. Open the folder as a volume series in
#@markdown ChimeraX for a publication movie; a quick inline slice-animation preview is shown here.
#@markdown (Graph paths can be long — rendering time scales with the number of points.)
#@markdown Pixel size (Å/px) — **`0` resolves it from `ctf.pkl`** (`eval_vol` itself defaults to 1).
apix = 0  #@param {type:"number"}
preview_inline = True  #@param {type:"boolean"}

import os, glob
outdir = os.environ["CRYODRGN_OUTDIR"]
zpath = os.environ["CRYODRGN_ZPATH"]
weights = os.path.join(outdir, "weights.pkl")
config = os.path.join(outdir, "config.yaml")
vol_dir = os.path.join(os.path.dirname(zpath), "volumes")

# See 9.1 — eval_vol never infers A/px from the CTF, so do it here (analyze.py:462-488).
use_apix, apix_src = float(apix), "form field"
if use_apix <= 0:
    import cryodrgn.config
    from cryodrgn import utils as _cdutils
    _cfg = cryodrgn.config.load(config)
    _ctf = _cfg["dataset_args"].get("ctf")
    use_apix, apix_src = 1.0, "fallback — no ctf.pkl in config.yaml"
    if _ctf and os.path.exists(_ctf):
        _cp = _cdutils.load_pkl(_ctf)
        _ap = set(_cp[:, 1])
        if len(_ap) > 1:
            apix_src = "fallback — multiple optics groups"
        else:
            use_apix = round(tuple(_ap)[0] * tuple(set(_cp[:, 0]))[0]
                             / (_cfg["lattice_args"]["D"] - 1), 6)
            apix_src = "from ctf.pkl"
print(f"A/px = {use_apix}  ({apix_src})")

cmd = f'cryodrgn eval_vol "{weights}" --config "{config}" --zfile "{zpath}" -o "{vol_dir}"'
cmd += f" --Apix {use_apix}"
print("$", cmd, "\n")
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")

vols = sorted(glob.glob(os.path.join(vol_dir, "*.mrc")))
print(f"\n✅ {len(vols)} volumes → {vol_dir}")

if preview_inline and vols:
    import matplotlib.pyplot as plt
    from matplotlib import animation
    from IPython.display import HTML, display
    from cryodrgn.mrcfile import parse_mrc
    step = max(1, len(vols) // 60)  # cap the preview at ~60 frames
    slices = []
    for v in vols[::step]:
        vol, _ = parse_mrc(v)
        slices.append(vol[vol.shape[0] // 2].copy())  # keep only the central slice
        del vol
    fig, ax = plt.subplots(figsize=(4, 4)); ax.axis("off")
    im = ax.imshow(slices[0], cmap="Greys_r")
    def _upd(i):
        im.set_data(slices[i]); ax.set_title(f"frame {i + 1}/{len(slices)}"); return [im]
    anim = animation.FuncAnimation(fig, _upd, frames=len(slices), interval=150, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

In [ ]:
#@title 9.4 · (Advanced) Conformational landscape analysis { display-mode: "form" }
#@markdown `analyze_landscape` maps the whole landscape: it sketches many volumes, auto-builds a
#@markdown mask, then does volume-space PCA + clustering. Heavier than `analyze` (several minutes);
#@markdown volumes are downsampled to `-d` for speed. For the full landscape, follow with
#@markdown `cryodrgn analyze_landscape_full` on the command line.
epoch = -1  #@param {type:"integer"}
sketch_size = 1000  #@param {type:"integer"}
downsample = 128  #@param {type:"integer"}
#@markdown Pixel size (Å/px) — **`0` resolves it from `ctf.pkl`**. This one is *not* cosmetic:
#@markdown `analyze_landscape`'s `--dilate` (5) and `--cosine-edge` are in **Ångströms** and are
#@markdown converted to voxels using it, so a wrong value changes the mask the PCA runs on.
apix = 0  #@param {type:"number"}

import os
outdir = os.environ["CRYODRGN_OUTDIR"]
if int(epoch) < 0:
    epoch = int(os.environ.get("CRYODRGN_EPOCH", "-1"))
    if epoch < 0:
        raise ValueError("Run Step 7 (analyze) first, or set epoch explicitly.")

# analyze_landscape declares --Apix with default=1 and never consults the CTF; resolve it the
# way `cryodrgn analyze` does (analyze.py:462-488).
import cryodrgn.config
from cryodrgn import utils as _cdutils
_cfg = cryodrgn.config.load(os.path.join(outdir, "config.yaml"))
_train_box = _cfg["lattice_args"]["D"] - 1
use_apix, apix_src = float(apix), "form field"
if use_apix <= 0:
    _ctf = _cfg["dataset_args"].get("ctf")
    use_apix, apix_src = 1.0, "fallback — no ctf.pkl in config.yaml"
    if _ctf and os.path.exists(_ctf):
        _cp = _cdutils.load_pkl(_ctf)
        _ap = set(_cp[:, 1])
        if len(_ap) > 1:
            apix_src = "fallback — multiple optics groups"
        else:
            use_apix = round(tuple(_ap)[0] * tuple(set(_cp[:, 0]))[0] / _train_box, 6)
            apix_src = "from ctf.pkl"
# -d decodes onto a coarser lattice (eval_vol.py:232-236) but neither eval_vol nor
# analyze_landscape rescales Apix for it, so do that here. No-op when -d == the training box.
if int(downsample) > 0 and int(downsample) != _train_box:
    use_apix = round(use_apix * _train_box / int(downsample), 6)
    apix_src += f", rescaled for -d {int(downsample)}"
print(f"A/px = {use_apix}  ({apix_src})")

cmd = (f'cryodrgn analyze_landscape "{outdir}" {int(epoch)} '
       f'-N {int(sketch_size)} -d {int(downsample)}')
cmd += f" --Apix {use_apix}"
print("$", cmd, "\n" + "=" * 70)
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    # -9 / 137 is SIGKILL. Nothing in the log explains it because the process was killed
    # outright, and on Colab that is essentially always the OOM killer.
    if _rc in (-9, 137):
        raise MemoryError(
            "The training process was KILLED (SIGKILL), which on Colab means it ran out of\n"
            "  RAM. There is no traceback because nothing was allowed to raise.\n"
            "  If the log stops at 'Loading dataset', the eager load was too big: check that\n"
            "  --lazy appears in the command above. If it does not, you are running an older\n"
            "  notebook — this cell decides --lazy from measured RAM and prints its reasoning\n"
            "  ('eager RAM : ...') before launching. Re-download the notebook.")
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print("=" * 70 + f"\n✅ Landscape → {outdir}/landscape.{int(epoch)}")

## 10 · Save & download results

Almost everything is already durable on Drive: the downsampled stack, `pose.pkl` and `ctf.pkl`
(Step 4), and all model/analysis outputs (Steps 6–9 write there directly). The only local-only
artifact is the optional back-projection map from Step 5 — copy it over here, or download any
results folder to your computer.

In [ ]:
#@title 10.1 · Back up the sanity-check map to Drive { display-mode: "form" }
#@markdown Copies the optional back-projection map (Step 5, which lives on local scratch) into
#@markdown your Drive project folder. Your stack / pose / ctf / model outputs are already there.
import os, shutil
WORK_DIR = os.environ["CRYODRGN_WORK_DIR"]
DRIVE_DIR = os.environ["CRYODRGN_DRIVE_DIR"]

bp = os.path.join(WORK_DIR, "backproject")
if os.path.isdir(bp):
    shutil.copytree(bp, os.path.join(DRIVE_DIR, "backproject"), dirs_exist_ok=True)
    print("✅ backproject/ → Drive")
else:
    print("No local back-projection map to copy (Step 5 not run this session).")

print(f"\nEverything durable is in: {DRIVE_DIR}")

In [ ]:
#@title 10.2 · (Optional) Zip an analysis folder and download it { display-mode: "form" }
#@markdown Bundles `analyze.<epoch>/` (plots + volumes) or `convergence.<epoch>/` (the 7.5
#@markdown diagnostics) into a zip and downloads it to your computer.
folder = "analyze"  #@param ["analyze", "convergence"]
#@markdown Epoch to bundle — **-1** picks the newest folder of that kind.
epoch = -1  #@param {type:"integer"}
import os, re, glob, shutil
from google.colab import files

outdir = os.environ["CRYODRGN_OUTDIR"]
if int(epoch) < 0:
    found = [int(m.group(1)) for p in glob.glob(os.path.join(outdir, f"{folder}.*"))
             for m in [re.search(rf"{folder}\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(
            f"No {folder}.N/ folder in {outdir} — "
            + ("run Step 7 first." if folder == "analyze" else "run cell 7.5 first.")
        )
    epoch = max(found)
    print(f"Auto-selected {folder}.{epoch}")
epoch = int(epoch)
adir = os.path.join(outdir, f"{folder}.{epoch}")
if not os.path.isdir(adir):
    raise FileNotFoundError(f"{adir} not found.")

zip_base = os.path.join(os.environ["CRYODRGN_WORK_DIR"], f"{folder}.{epoch}")
print("Zipping", adir, "...")
shutil.make_archive(zip_base, "zip", adir)
print("Starting download of", zip_base + ".zip")
files.download(zip_base + ".zip")

## 11 · Tips, troubleshooting & next steps

**Colab session limits**
- Free Colab disconnects after idle time and caps total runtime. Because per-epoch checkpoints
  are written to Drive, you can always resume with **cell 6.2** (`--load`).
- For big datasets or `D=256`, use **Colab Pro/Pro+** (A100/L4, longer sessions, more RAM).

**Common issues**
- *Back-projection / volumes look like noise* → poses or CTF likely mis-parsed. Re-check the box
  size `-D` (must be the **consensus** box, not the downsampled one) and toggle `uninvert_data`.
- *`CUDA out of memory`* → downsample to a smaller box (128), lower the batch size, or use a bigger GPU.
- *`.star`/`.cs` paths broken* → set **`datadir`** (cell 3.1) to the folder holding the `.mrcs`.
- *Slow training* → `downsample` (4.1) already writes the training stack to fast local disk; keep
  `D=128` for the first pass and only move to `D=256` once results look good.
- *Out of disk on `/content`* → free space by deleting the local downsampled stack, or work at `D=128`.

**Has it converged? Is it overfitting?**
- *Converged* — **cells 7.4–7.6**. 7.4 plots loss/KLD from `run.log` and is safe to run mid-training;
  7.5 runs `analyze_convergence` (latent-shift magnitude decaying to ~0 is the clearest signal).
- *Overfitting* — `train_vae` keeps **no held-out set**, so the loss curve cannot show it. The only
  honest measurement is a validation loss you build yourself: train on a subset with
  `--ind train_ind.pkl`, then score the excluded particles at several checkpoints with
  `cryodrgn eval_images STACK OUTDIR/weights.N.pkl -c OUTDIR/config.yaml --poses pose.pkl
  --ctf ctf.pkl --ind val_ind.pkl -o val.N.pkl --out-z val_z.N.pkl` (add `--uninvert-data` if you
  trained with it). It writes `{loss, recon, kld}` computed by the *same* `loss_function` as
  training, so plot `recon` beside the training curve — divergence is overfitting. One forward pass
  per checkpoint, so a 5 % held-out set costs minutes.
- Watch the **KLD** term either way: collapsing toward 0 means the latent is unused (you are fitting
  a consensus map); climbing while reconstruction falls means the latent is absorbing per-particle
  noise, which is what raising `zdim` too far looks like.

**Going further** (all available as `cryodrgn ...` commands)
- *Particle filtering* — see **Step 8**: deterministic cuts in Colab, or export a bundle for the
  interactive `cryodrgn filter` lasso locally.
- *Trajectories & landscape* — see **Step 9** (9.2–9.4): graph/PC/direct traversals rendered to a
  volume series, plus `analyze_landscape` (extend it with `analyze_landscape_full` on the CLI).
- `cryodrgn abinit_homo` / `abinit_het` — *ab-initio* reconstruction (no consensus poses needed).
- **cryoDRGN-ET** — heterogeneous subtomogram averaging for cryo-ET.

📖 Full walkthroughs: <https://ez-lab.gitbook.io/cryodrgn/> · Questions/bugs → [GitHub issues](https://github.com/ml-struct-bio/cryodrgn/issues).

---
*Volumes are best inspected in [ChimeraX](https://www.cgl.ucsf.edu/chimerax) — download the `.mrc`
files from your Drive project folder and open them there for publication-quality figures.*